# Medical Report Explainer

## 1) Install

In [1]:
!pip install -q transformers==4.52.4 accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-core langchain-classic langchain-huggingface
!pip install -q faiss-cpu sentence-transformers pypdf streamlit pyngrok streamlit-geolocation requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 12.8 MB/s eta 0:00:0000:01
   ━━━

## 2) Project folder

In [2]:
!mkdir -p medical_explainer

## 3) Knowledge base
A JSON file with the reference range, a short meaning, the English and Arabic explanation and the related tests for each test. FAISS searches it later.


In [3]:
%%writefile medical_explainer/knowledge_base.json
[
 {
  "key": "hemoglobin",
  "name": "Hemoglobin",
  "aliases": [
   "hemoglobin",
   "haemoglobin",
   "hgb",
   "hb",
   "hb level"
  ],
  "category": "Complete Blood Count",
  "unit": "g/dL",
  "range": {
   "male": [
    13.5,
    17.5
   ],
   "female": [
    12.0,
    15.5
   ],
   "general": [
    12.0,
    17.5
   ]
  },
  "critical": {
   "low": 7.0,
   "high": 20.0
  },
  "measures": "The protein inside red blood cells that carries oxygen around the body.",
  "low": "Low levels are called anemia. Common reasons include iron deficiency, blood loss, vitamin B12 or folate deficiency, and long-term illness. It can cause tiredness, paleness and shortness of breath.",
  "high": "High levels can be linked to dehydration, smoking, living at high altitude, lung disease, or conditions that increase red blood cell production.",
  "why": "Checks for anemia and shows how well your blood can carry oxygen. It is part of almost every routine check-up.",
  "prep": "No fasting needed. Dehydration can make it look higher, while extra fluids or pregnancy can make it look lower.",
  "related": [
   "hematocrit",
   "rbc",
   "mcv",
   "ferritin",
   "iron",
   "vitamin_b12",
   "folate"
  ],
  "measures_ar": "البروتين الموجود داخل كرات الدم الحمراء والذي ينقل الأكسجين في الجسم.",
  "why_ar": "يكشف عن الأنيميا (فقر الدم) ويوضح قدرة الدم على نقل الأكسجين، وهو جزء من أغلب الفحوصات الدورية.",
  "prep_ar": "لا يحتاج إلى صيام. الجفاف قد يجعله يبدو أعلى، وزيادة السوائل أو الحمل قد تجعله يبدو أقل."
 },
 {
  "key": "hematocrit",
  "name": "Hematocrit",
  "aliases": [
   "hematocrit",
   "haematocrit",
   "hct",
   "pcv",
   "packed cell volume"
  ],
  "category": "Complete Blood Count",
  "unit": "%",
  "range": {
   "male": [
    41.0,
    53.0
   ],
   "female": [
    36.0,
    46.0
   ],
   "general": [
    36.0,
    53.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The percentage of your blood volume that is made up of red blood cells.",
  "low": "It usually moves together with hemoglobin. Low values are seen in anemia, blood loss or overhydration.",
  "high": "High values can be seen with dehydration, smoking, or high altitude, and sometimes with conditions that increase red cell production.",
  "why": "Gives a second view of your red blood cells. Doctors read it together with hemoglobin.",
  "prep": "No fasting needed. Dehydration raises it; a lot of fluids or pregnancy can lower it.",
  "related": [
   "hemoglobin",
   "rbc",
   "mcv"
  ],
  "measures_ar": "النسبة المئوية من حجم الدم التي تشغلها كرات الدم الحمراء.",
  "why_ar": "يعطي صورة إضافية عن كرات الدم الحمراء ويُقرأ عادة مع الهيموجلوبين.",
  "prep_ar": "لا يحتاج إلى صيام. الجفاف يرفعه، والإكثار من السوائل أو الحمل قد يخفضه."
 },
 {
  "key": "rbc",
  "name": "Red Blood Cell Count",
  "aliases": [
   "rbc",
   "rbc count",
   "red blood cells",
   "red blood cell count",
   "red cell count",
   "erythrocytes"
  ],
  "category": "Complete Blood Count",
  "unit": "x10^6/uL",
  "range": {
   "male": [
    4.5,
    5.9
   ],
   "female": [
    4.1,
    5.1
   ],
   "general": [
    4.1,
    5.9
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The number of red blood cells, which carry oxygen.",
  "low": "Low counts are seen in anemia, blood loss, and some nutritional deficiencies or long-term illnesses.",
  "high": "High counts can be linked to dehydration, smoking, high altitude, or lung and heart conditions.",
  "why": "Counts red blood cells to help find the cause of anemia or of unusually high red cell levels.",
  "prep": "No fasting needed. Living at altitude, dehydration and smoking can raise the count.",
  "related": [
   "hemoglobin",
   "hematocrit",
   "mcv",
   "rdw"
  ],
  "measures_ar": "عدد كرات الدم الحمراء التي تحمل الأكسجين.",
  "why_ar": "يساعد على معرفة سبب الأنيميا أو ارتفاع كرات الدم الحمراء.",
  "prep_ar": "لا يحتاج إلى صيام. العيش في مناطق مرتفعة والجفاف والتدخين قد ترفع العدد."
 },
 {
  "key": "wbc",
  "name": "White Blood Cell Count",
  "aliases": [
   "wbc",
   "wbc count",
   "white blood cells",
   "white blood cell count",
   "white cell count",
   "tlc",
   "total leukocyte count",
   "total leucocyte count",
   "leukocytes"
  ],
  "category": "Complete Blood Count",
  "unit": "x10^3/uL",
  "range": {
   "general": [
    4.5,
    11.0
   ]
  },
  "critical": {
   "low": 2.0,
   "high": 30.0
  },
  "measures": "The number of white blood cells, which help fight infections.",
  "low": "Low counts can follow viral infections, some medicines, or bone marrow and immune conditions.",
  "high": "High counts are often seen with infections, inflammation, stress, smoking or some medicines, and rarely with blood disorders.",
  "why": "Shows how your immune system is doing and can point to infection, inflammation or other blood conditions.",
  "prep": "No fasting needed. A recent infection, hard exercise, stress and smoking can all change it.",
  "related": [
   "platelets",
   "crp",
   "esr"
  ],
  "measures_ar": "عدد كرات الدم البيضاء التي تحارب العدوى.",
  "why_ar": "يعكس نشاط جهاز المناعة وقد يشير إلى عدوى أو التهاب أو اضطرابات في الدم.",
  "prep_ar": "لا يحتاج إلى صيام. العدوى الحديثة والمجهود العنيف والتوتر والتدخين قد تغيّره."
 },
 {
  "key": "platelets",
  "name": "Platelet Count",
  "aliases": [
   "platelets",
   "platelet count",
   "plt",
   "platelet",
   "thrombocytes"
  ],
  "category": "Complete Blood Count",
  "unit": "x10^3/uL",
  "range": {
   "general": [
    150.0,
    450.0
   ]
  },
  "critical": {
   "low": 20.0,
   "high": 1000.0
  },
  "measures": "Small blood cells that help the blood to clot.",
  "low": "Low counts can be linked to viral infections, some medicines, immune conditions or bone marrow problems, and may increase bleeding or bruising.",
  "high": "High counts can occur with inflammation, infection, iron deficiency or after surgery, and rarely with bone marrow conditions.",
  "why": "Checks how well your blood can clot, especially before procedures or when there is easy bruising or bleeding.",
  "prep": "No fasting needed. A recent infection, pregnancy or some medicines can change the count, so tell your doctor what you take.",
  "related": [
   "wbc",
   "hemoglobin",
   "inr"
  ],
  "measures_ar": "خلايا دموية صغيرة تساعد الدم على التجلط.",
  "why_ar": "يفحص قدرة الدم على التجلط، خاصة قبل الإجراءات الطبية أو عند سهولة ظهور الكدمات أو النزيف.",
  "prep_ar": "لا يحتاج إلى صيام. العدوى الحديثة والحمل وبعض الأدوية قد تغيّر العدد، لذلك أخبر طبيبك بما تتناوله."
 },
 {
  "key": "mcv",
  "name": "MCV",
  "aliases": [
   "mcv",
   "mean corpuscular volume",
   "mean cell volume"
  ],
  "category": "Complete Blood Count",
  "unit": "fL",
  "range": {
   "general": [
    80.0,
    100.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The average size of your red blood cells.",
  "low": "Small red cells are commonly seen with iron deficiency or thalassemia trait.",
  "high": "Large red cells are commonly seen with vitamin B12 or folate deficiency, alcohol use, or thyroid problems.",
  "why": "Helps sort anemia into types, because small, normal or large red cells point to different causes.",
  "prep": "No fasting needed.",
  "related": [
   "hemoglobin",
   "mch",
   "rdw",
   "ferritin",
   "vitamin_b12",
   "folate"
  ],
  "measures_ar": "متوسط حجم كرات الدم الحمراء.",
  "why_ar": "يساعد على تصنيف الأنيميا، فصغر الكرات أو كبرها يشير إلى أسباب مختلفة.",
  "prep_ar": "لا يحتاج إلى صيام."
 },
 {
  "key": "mch",
  "name": "MCH",
  "aliases": [
   "mch",
   "mean corpuscular hemoglobin",
   "mean cell hemoglobin"
  ],
  "category": "Complete Blood Count",
  "unit": "pg",
  "range": {
   "general": [
    27.0,
    33.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The average amount of hemoglobin in each red blood cell.",
  "low": "Low values go along with small, pale red cells, often because of iron deficiency.",
  "high": "High values go along with large red cells, for example in vitamin B12 or folate deficiency.",
  "why": "Shows whether red cells carry enough hemoglobin. It is read together with MCV and MCHC.",
  "prep": "No fasting needed.",
  "related": [
   "mcv",
   "mchc",
   "hemoglobin"
  ],
  "measures_ar": "متوسط كمية الهيموجلوبين في كل كرة دم حمراء.",
  "why_ar": "يوضح إن كانت الكرات الحمراء تحمل كمية كافية من الهيموجلوبين، ويُقرأ مع MCV وMCHC.",
  "prep_ar": "لا يحتاج إلى صيام."
 },
 {
  "key": "mchc",
  "name": "MCHC",
  "aliases": [
   "mchc",
   "mean corpuscular hemoglobin concentration",
   "mean cell hemoglobin concentration"
  ],
  "category": "Complete Blood Count",
  "unit": "g/dL",
  "range": {
   "general": [
    32.0,
    36.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "How concentrated the hemoglobin is inside your red blood cells.",
  "low": "Low values are commonly seen in iron deficiency anemia.",
  "high": "High values are uncommon and can be a laboratory artifact or linked to some red cell conditions.",
  "why": "Adds detail on how full red cells are with hemoglobin, which helps classify anemia.",
  "prep": "No fasting needed.",
  "related": [
   "mch",
   "mcv",
   "hemoglobin"
  ],
  "measures_ar": "تركيز الهيموجلوبين داخل كرات الدم الحمراء.",
  "why_ar": "يضيف تفصيلاً عن مدى امتلاء الكرات بالهيموجلوبين، وهذا يساعد على تصنيف الأنيميا.",
  "prep_ar": "لا يحتاج إلى صيام."
 },
 {
  "key": "rdw",
  "name": "RDW",
  "aliases": [
   "rdw",
   "rdw cv",
   "red cell distribution width"
  ],
  "category": "Complete Blood Count",
  "unit": "%",
  "range": {
   "general": [
    11.5,
    14.5
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "How much your red blood cells vary in size.",
  "low": "Low values are usually not a concern.",
  "high": "High values mean the red cells vary a lot in size, which is common in iron, vitamin B12 or folate deficiency.",
  "why": "Shows how uneven your red cell sizes are, which can be an early sign of an iron, B12 or folate shortage.",
  "prep": "No fasting needed.",
  "related": [
   "mcv",
   "ferritin",
   "vitamin_b12",
   "folate",
   "hemoglobin"
  ],
  "measures_ar": "مدى اختلاف أحجام كرات الدم الحمراء عن بعضها.",
  "why_ar": "يوضح مدى تفاوت أحجام الكرات، وقد يكون علامة مبكرة على نقص الحديد أو B12 أو حمض الفوليك.",
  "prep_ar": "لا يحتاج إلى صيام."
 },
 {
  "key": "glucose_fasting",
  "name": "Fasting Blood Glucose",
  "aliases": [
   "fasting blood sugar",
   "fbs",
   "fasting glucose",
   "glucose fasting",
   "fasting plasma glucose",
   "fpg",
   "fasting blood glucose",
   "glucose",
   "blood glucose",
   "blood sugar"
  ],
  "category": "Blood Sugar",
  "unit": "mg/dL",
  "range": {
   "general": [
    70.0,
    99.0
   ]
  },
  "critical": {
   "low": 50.0,
   "high": 400.0
  },
  "measures": "The amount of sugar in your blood after not eating for at least 8 hours.",
  "low": "Low sugar can happen after skipping meals, with some diabetes medicines, or in certain hormone or liver conditions, and can cause shakiness and sweating.",
  "high": "Values from 100 to 125 mg/dL are in the prediabetes range, and 126 mg/dL or more on repeated tests is in the diabetes range, but only a doctor can make a diagnosis. Stress, illness and recent meals can also raise it.",
  "note": "Assumes the sample was taken after fasting. If it was a random sample, ask your doctor how to interpret it.",
  "why": "Screens for prediabetes and diabetes and helps follow how well sugar is controlled.",
  "prep": "Fast for about 8 hours (water is fine). A recent meal, stress, illness or poor sleep can raise it. Do not change your usual medicines unless your doctor tells you.",
  "related": [
   "hba1c",
   "glucose_random",
   "triglycerides"
  ],
  "measures_ar": "كمية السكر في الدم بعد صيام لا يقل عن 8 ساعات.",
  "why_ar": "يستخدم للكشف عن مرحلة ما قبل السكري والسكري ولمتابعة انضباط السكر.",
  "prep_ar": "الصيام نحو 8 ساعات (يمكن شرب الماء). الوجبة الأخيرة والتوتر والمرض وقلة النوم قد ترفعه. لا تغيّر أدويتك المعتادة إلا بتوجيه من طبيبك."
 },
 {
  "key": "glucose_random",
  "name": "Random or Post-Meal Blood Glucose",
  "aliases": [
   "random blood sugar",
   "rbs",
   "random glucose",
   "random blood glucose",
   "glucose random",
   "postprandial glucose",
   "2 hour postprandial glucose",
   "ppbs",
   "pp blood sugar",
   "2hpp"
  ],
  "category": "Blood Sugar",
  "unit": "mg/dL",
  "range": {
   "general": [
    70.0,
    140.0
   ]
  },
  "critical": {
   "low": 50.0,
   "high": 400.0
  },
  "measures": "Blood sugar measured at any time of day, or two hours after a meal.",
  "low": "Low sugar can happen after skipping meals, with some diabetes medicines, or in certain hormone or liver conditions.",
  "high": "Values above 140 mg/dL after a meal, or 200 mg/dL or more at random, can suggest abnormal sugar control, but a doctor needs to confirm this with other tests.",
  "why": "Gives a snapshot of your sugar at the moment of testing. Useful when fasting is not possible or when symptoms appear.",
  "prep": "No fasting needed, but note when you last ate, because the timing of the last meal changes the result a lot.",
  "related": [
   "glucose_fasting",
   "hba1c"
  ],
  "measures_ar": "مستوى السكر في أي وقت من اليوم أو بعد ساعتين من الوجبة.",
  "why_ar": "يعطي لقطة عن السكر لحظة التحليل، ويفيد عندما يتعذر الصيام أو عند ظهور أعراض.",
  "prep_ar": "لا يحتاج إلى صيام، لكن دوّن وقت آخر وجبة لأنها تؤثر كثيراً على النتيجة."
 },
 {
  "key": "hba1c",
  "name": "HbA1c",
  "aliases": [
   "hba1c",
   "a1c",
   "hemoglobin a1c",
   "glycated hemoglobin",
   "glycosylated hemoglobin",
   "glycohemoglobin",
   "hgb a1c",
   "hb a1c"
  ],
  "category": "Blood Sugar",
  "unit": "%",
  "range": {
   "general": [
    4.0,
    5.6
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "Your average blood sugar level over roughly the past 3 months.",
  "low": "Low values are uncommon and can be seen with anemia or recent blood loss, which can affect the test.",
  "high": "5.7 to 6.4% is the prediabetes range and 6.5% or more is the diabetes range, but only a doctor can make a diagnosis.",
  "why": "Shows your average sugar over about 3 months and does not depend on your last meal. Used to screen for and follow diabetes.",
  "prep": "No fasting needed. Anemia, recent blood loss or transfusion and some hemoglobin variants can make it unreliable.",
  "related": [
   "glucose_fasting",
   "glucose_random",
   "hemoglobin"
  ],
  "measures_ar": "متوسط مستوى السكر في الدم خلال نحو 3 شهور.",
  "why_ar": "يوضح متوسط السكر خلال 3 شهور تقريباً ولا يتأثر بآخر وجبة، ويُستخدم للكشف عن السكري ومتابعته.",
  "prep_ar": "لا يحتاج إلى صيام. الأنيميا أو النزيف الحديث أو نقل الدم قد تجعل النتيجة غير دقيقة."
 },
 {
  "key": "total_cholesterol",
  "name": "Total Cholesterol",
  "aliases": [
   "total cholesterol",
   "cholesterol",
   "cholesterol total",
   "t cholesterol",
   "tc",
   "serum cholesterol"
  ],
  "category": "Lipid Profile",
  "unit": "mg/dL",
  "range": {
   "general": [
    0.0,
    200.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The total amount of cholesterol, a fatty substance, in your blood.",
  "low": "Very low values are uncommon and may be linked to malnutrition, or to liver or thyroid conditions.",
  "high": "200 to 239 mg/dL is borderline high and 240 or more is high. It raises heart disease risk over time and is often improved with diet and activity, and sometimes medication decided by a doctor.",
  "why": "Gives a first look at blood fats and heart-related risk. It is interpreted together with LDL, HDL and triglycerides.",
  "prep": "Follow the lab's fasting instructions. Recent illness, pregnancy and fast weight change can shift it.",
  "related": [
   "ldl",
   "hdl",
   "triglycerides"
  ],
  "measures_ar": "إجمالي الدهون من نوع الكوليسترول في الدم.",
  "why_ar": "يعطي نظرة أولى عن دهون الدم ومخاطر القلب، ويُقرأ مع LDL وHDL والدهون الثلاثية.",
  "prep_ar": "اتبع تعليمات المعمل بخصوص الصيام. المرض الحديث والحمل وتغيّر الوزن السريع قد يغيّرون النتيجة."
 },
 {
  "key": "ldl",
  "name": "LDL Cholesterol",
  "aliases": [
   "ldl",
   "ldl cholesterol",
   "ldl c",
   "low density lipoprotein",
   "ldl direct",
   "ldl calculated",
   "ldl cholesterol calculated"
  ],
  "category": "Lipid Profile",
  "unit": "mg/dL",
  "range": {
   "general": [
    0.0,
    100.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The so-called bad cholesterol, which can build up in the walls of arteries.",
  "low": "Low values are generally good and rarely a concern.",
  "high": "High LDL raises the long-term risk of heart disease. Your doctor will consider it together with your other risk factors.",
  "why": "Often called the bad cholesterol. It is one of the main numbers used to estimate heart and blood vessel risk.",
  "prep": "Follow the lab's fasting instructions. If triglycerides are measured at the same time, fasting for 9 to 12 hours is common.",
  "related": [
   "total_cholesterol",
   "hdl",
   "triglycerides"
  ],
  "measures_ar": "الكوليسترول الذي يُعرف بالضار، وهو ينقل الكوليسترول إلى جدران الأوعية.",
  "why_ar": "من أهم الأرقام المستخدمة لتقدير مخاطر القلب والأوعية الدموية.",
  "prep_ar": "اتبع تعليمات المعمل بخصوص الصيام. عند قياس الدهون الثلاثية معه يكون الصيام من 9 إلى 12 ساعة شائعاً."
 },
 {
  "key": "hdl",
  "name": "HDL Cholesterol",
  "aliases": [
   "hdl",
   "hdl cholesterol",
   "hdl c",
   "high density lipoprotein"
  ],
  "category": "Lipid Profile",
  "unit": "mg/dL",
  "range": {
   "male": [
    40.0,
    null
   ],
   "female": [
    50.0,
    null
   ],
   "general": [
    40.0,
    null
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The so-called good cholesterol, which helps remove other cholesterol from the blood.",
  "low": "Low HDL is linked to higher heart risk and is often related to inactivity, smoking, excess weight or high blood sugar.",
  "high": "High HDL is generally considered protective.",
  "why": "Often called the good cholesterol. Higher levels are generally linked with lower heart risk.",
  "prep": "Follow the lab's fasting instructions. Regular exercise and stopping smoking are linked with higher HDL.",
  "related": [
   "total_cholesterol",
   "ldl",
   "triglycerides"
  ],
  "measures_ar": "الكوليسترول الذي يُعرف بالنافع، ويساعد على إزالة الكوليسترول الزائد من الدم.",
  "why_ar": "الارتفاع فيه يرتبط عموماً بانخفاض مخاطر أمراض القلب.",
  "prep_ar": "اتبع تعليمات المعمل بخصوص الصيام. الرياضة المنتظمة والإقلاع عن التدخين يرتبطان بارتفاع HDL."
 },
 {
  "key": "triglycerides",
  "name": "Triglycerides",
  "aliases": [
   "triglycerides",
   "triglyceride",
   "tg",
   "tgs",
   "serum triglycerides"
  ],
  "category": "Lipid Profile",
  "unit": "mg/dL",
  "range": {
   "general": [
    0.0,
    150.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A type of fat in the blood that comes from food and is also made by the liver.",
  "low": "Low values are usually not a concern.",
  "high": "High values are often linked to sugary or fatty diets, alcohol, excess weight, diabetes or low thyroid function. Very high levels can irritate the pancreas.",
  "why": "Measures a blood fat that rises with sugar, alcohol and extra calories. Helps assess heart risk and, when very high, pancreas risk.",
  "prep": "Fast for 9 to 12 hours and avoid alcohol for 24 hours before the test. A recent fatty meal can raise it a lot.",
  "related": [
   "total_cholesterol",
   "ldl",
   "hdl",
   "glucose_fasting"
  ],
  "measures_ar": "نوع من دهون الدم يرتفع مع السكر والكحول والسعرات الزائدة.",
  "why_ar": "يساعد على تقدير مخاطر القلب، وإذا ارتفع جداً فقد يؤثر على البنكرياس.",
  "prep_ar": "الصيام من 9 إلى 12 ساعة وتجنب الكحول 24 ساعة قبل التحليل. الوجبة الدسمة الأخيرة قد ترفعه كثيراً."
 },
 {
  "key": "alt",
  "name": "ALT (SGPT)",
  "aliases": [
   "alt",
   "alt sgpt",
   "sgpt",
   "alanine aminotransferase",
   "alanine transaminase",
   "alt gpt",
   "gpt"
  ],
  "category": "Liver Function",
  "unit": "U/L",
  "range": {
   "male": [
    7.0,
    55.0
   ],
   "female": [
    7.0,
    45.0
   ],
   "general": [
    7.0,
    55.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A liver enzyme that rises when liver cells are irritated or damaged.",
  "low": "Low values are usually not a concern.",
  "high": "Mild rises are common with fatty liver, alcohol, some medicines and supplements, viral hepatitis or strenuous exercise. Larger rises need medical review.",
  "why": "Checks for irritation or damage of liver cells.",
  "prep": "Usually no fasting. Hard exercise, alcohol and some medicines or supplements can raise it, so tell your doctor what you take.",
  "related": [
   "ast",
   "ggt",
   "alp",
   "bilirubin_total"
  ],
  "measures_ar": "إنزيم يوجد أساساً في الكبد ويرتفع في الدم عند تهيج خلايا الكبد.",
  "why_ar": "يفحص وجود تهيج أو تلف في خلايا الكبد.",
  "prep_ar": "غالباً لا يحتاج إلى صيام. المجهود العنيف والكحول وبعض الأدوية أو المكملات قد ترفعه، لذلك أخبر طبيبك بما تتناوله."
 },
 {
  "key": "ast",
  "name": "AST (SGOT)",
  "aliases": [
   "ast",
   "ast sgot",
   "sgot",
   "aspartate aminotransferase",
   "aspartate transaminase",
   "ast got",
   "got"
  ],
  "category": "Liver Function",
  "unit": "U/L",
  "range": {
   "male": [
    8.0,
    48.0
   ],
   "female": [
    8.0,
    43.0
   ],
   "general": [
    8.0,
    48.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "An enzyme found in the liver, muscles and heart.",
  "low": "Low values are usually not a concern.",
  "high": "It can rise with liver conditions, muscle injury, intense exercise, alcohol or some medicines.",
  "why": "Checks the liver, but it also comes from muscle and heart, so it is read alongside ALT.",
  "prep": "Usually no fasting. Hard exercise in the previous 1 to 2 days, alcohol and muscle injury can raise it.",
  "related": [
   "alt",
   "ggt",
   "alp"
  ],
  "measures_ar": "إنزيم يوجد في الكبد والعضلات والقلب.",
  "why_ar": "يفحص الكبد، لكنه يأتي أيضاً من العضلات والقلب لذلك يُقرأ مع ALT.",
  "prep_ar": "غالباً لا يحتاج إلى صيام. المجهود العنيف خلال اليوم أو اليومين السابقين والكحول وإصابات العضلات قد ترفعه."
 },
 {
  "key": "alp",
  "name": "Alkaline Phosphatase (ALP)",
  "aliases": [
   "alp",
   "alkaline phosphatase",
   "alk phos",
   "alkaline phosphatase alp"
  ],
  "category": "Liver Function",
  "unit": "U/L",
  "range": {
   "general": [
    44.0,
    147.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "An enzyme found mainly in the liver and bones.",
  "low": "Low values are uncommon and may be linked to nutritional deficiencies.",
  "high": "It can be raised by bile duct or liver conditions, bone growth in children, bone diseases, or pregnancy.",
  "why": "Looks at bile flow and bone health. It rises with some bile duct or bone conditions.",
  "prep": "Fasting may be requested because a fatty meal can raise it slightly. Growing children and pregnancy normally have higher levels.",
  "related": [
   "ggt",
   "alt",
   "calcium",
   "bilirubin_total"
  ],
  "measures_ar": "إنزيم يوجد في القنوات الصفراوية والعظام.",
  "why_ar": "يفحص تدفق العصارة الصفراوية وصحة العظام، ويرتفع في بعض أمراض القنوات الصفراوية أو العظام.",
  "prep_ar": "قد يُطلب الصيام لأن الوجبة الدسمة ترفعه قليلاً. الأطفال في فترة النمو والحوامل مستواهم الطبيعي أعلى."
 },
 {
  "key": "ggt",
  "name": "GGT",
  "aliases": [
   "ggt",
   "gamma gt",
   "gamma glutamyl transferase",
   "gamma glutamyl transpeptidase",
   "gamma gt ggt"
  ],
  "category": "Liver Function",
  "unit": "U/L",
  "range": {
   "male": [
    8.0,
    61.0
   ],
   "female": [
    5.0,
    36.0
   ],
   "general": [
    8.0,
    61.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "An enzyme found in the liver and bile ducts.",
  "low": "Low values are usually not a concern.",
  "high": "It often rises with alcohol use, fatty liver, bile duct problems or some medicines.",
  "why": "A sensitive marker of bile duct and liver irritation. It helps tell whether a raised ALP comes from the liver or from bone.",
  "prep": "Avoid alcohol before the test. Some medicines can raise it.",
  "related": [
   "alp",
   "alt",
   "ast"
  ],
  "measures_ar": "إنزيم يوجد في الكبد والقنوات الصفراوية.",
  "why_ar": "مؤشر حساس لتهيج الكبد والقنوات الصفراوية، ويساعد على معرفة هل ارتفاع ALP مصدره الكبد أم العظام.",
  "prep_ar": "تجنب الكحول قبل التحليل. بعض الأدوية قد ترفعه."
 },
 {
  "key": "bilirubin_total",
  "name": "Total Bilirubin",
  "aliases": [
   "total bilirubin",
   "bilirubin",
   "bilirubin total",
   "t bilirubin",
   "serum bilirubin",
   "tbil"
  ],
  "category": "Liver Function",
  "unit": "mg/dL",
  "range": {
   "general": [
    0.1,
    1.2
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A yellow substance made when red blood cells break down, which the liver clears from the body.",
  "low": "Low values are usually not a concern.",
  "high": "High values can cause yellowing of the skin or eyes and may be linked to Gilbert syndrome, liver or bile duct problems, or increased breakdown of red blood cells.",
  "why": "Checks how well the liver processes old red cells and drains bile. High levels can turn the skin and eyes yellow.",
  "prep": "Fasting for many hours can raise it a little, so follow the lab's instructions.",
  "related": [
   "bilirubin_direct",
   "bilirubin_indirect",
   "alt",
   "ast",
   "alp",
   "hemoglobin"
  ],
  "measures_ar": "صبغة صفراء تنتج من تكسر كرات الدم الحمراء القديمة ويتخلص منها الكبد.",
  "why_ar": "يفحص قدرة الكبد على معالجة كرات الدم القديمة وتصريف العصارة الصفراوية، وارتفاعه الشديد قد يسبب اصفرار الجلد والعينين.",
  "prep_ar": "الصيام لساعات طويلة قد يرفعه قليلاً، لذلك اتبع تعليمات المعمل."
 },
 {
  "key": "bilirubin_direct",
  "name": "Direct Bilirubin",
  "aliases": [
   "direct bilirubin",
   "bilirubin direct",
   "conjugated bilirubin",
   "d bilirubin",
   "dbil"
  ],
  "category": "Liver Function",
  "unit": "mg/dL",
  "range": {
   "general": [
    0.0,
    0.3
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The part of bilirubin that has already been processed by the liver.",
  "low": "Low values are usually not a concern.",
  "high": "A higher direct fraction points more toward liver or bile duct causes.",
  "why": "Helps separate liver or bile-flow problems from red cell breakdown as the cause of a high total bilirubin.",
  "prep": "Same preparation as total bilirubin.",
  "related": [
   "bilirubin_total",
   "bilirubin_indirect",
   "alp",
   "ggt"
  ],
  "measures_ar": "البيليروبين بعد أن يعالجه الكبد ويجهزه للتخلص منه.",
  "why_ar": "يساعد على التفريق بين مشاكل الكبد أو تصريف العصارة وبين تكسر كرات الدم كسبب لارتفاع البيليروبين الكلي.",
  "prep_ar": "نفس تجهيز البيليروبين الكلي."
 },
 {
  "key": "bilirubin_indirect",
  "name": "Indirect Bilirubin",
  "aliases": [
   "indirect bilirubin",
   "bilirubin indirect",
   "unconjugated bilirubin"
  ],
  "category": "Liver Function",
  "unit": "mg/dL",
  "range": {
   "general": [
    0.1,
    0.8
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The part of bilirubin that has not yet been processed by the liver.",
  "low": "Low values are usually not a concern.",
  "high": "A higher indirect fraction can be linked to increased breakdown of red blood cells or to Gilbert syndrome.",
  "why": "Usually calculated as total minus direct bilirubin. A high value points more toward red cell breakdown or how the liver takes up bilirubin.",
  "prep": "Same preparation as total bilirubin.",
  "related": [
   "bilirubin_total",
   "bilirubin_direct",
   "hemoglobin"
  ],
  "measures_ar": "البيليروبين قبل أن يعالجه الكبد.",
  "why_ar": "يُحسب غالباً بطرح المباشر من الكلي، وارتفاعه يشير أكثر إلى تكسر كرات الدم أو طريقة التقاط الكبد للبيليروبين.",
  "prep_ar": "نفس تجهيز البيليروبين الكلي."
 },
 {
  "key": "albumin",
  "name": "Albumin",
  "aliases": [
   "albumin",
   "serum albumin",
   "alb"
  ],
  "category": "Liver Function",
  "unit": "g/dL",
  "range": {
   "general": [
    3.5,
    5.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A protein made by the liver that keeps fluid inside blood vessels and carries substances around the body.",
  "low": "Low values can be linked to liver or kidney disease, poor nutrition, or long-term inflammation.",
  "high": "High values are usually due to dehydration.",
  "why": "Shows how well the liver makes protein and gives a picture of nutrition. Kidneys and long-term illness can affect it too.",
  "prep": "No fasting needed. Dehydration can raise it and overhydration or lying down can lower it.",
  "related": [
   "total_protein",
   "alt",
   "creatinine"
  ],
  "measures_ar": "أهم بروتين في بلازما الدم، ويصنعه الكبد.",
  "why_ar": "يوضح قدرة الكبد على تصنيع البروتين ويعطي فكرة عن التغذية، وتتأثر به الكلى والأمراض المزمنة أيضاً.",
  "prep_ar": "لا يحتاج إلى صيام. الجفاف قد يرفعه، والإكثار من السوائل أو الاستلقاء قد يخفضه."
 },
 {
  "key": "total_protein",
  "name": "Total Protein",
  "aliases": [
   "total protein",
   "protein total",
   "serum protein",
   "tp"
  ],
  "category": "Liver Function",
  "unit": "g/dL",
  "range": {
   "general": [
    6.0,
    8.3
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The total amount of albumin and other proteins in the blood.",
  "low": "Low values can be linked to liver or kidney disease or poor nutrition.",
  "high": "High values can be linked to dehydration or to long-term inflammation.",
  "why": "A general check of nutrition and of how the liver and kidneys handle protein.",
  "prep": "No fasting needed. Dehydration can raise it.",
  "related": [
   "albumin",
   "alt"
  ],
  "measures_ar": "إجمالي البروتينات في الدم (الألبومين والجلوبيولين).",
  "why_ar": "فحص عام للتغذية ولطريقة تعامل الكبد والكلى مع البروتين.",
  "prep_ar": "لا يحتاج إلى صيام. الجفاف قد يرفعه."
 },
 {
  "key": "creatinine",
  "name": "Creatinine",
  "aliases": [
   "creatinine",
   "serum creatinine",
   "creat",
   "s creatinine"
  ],
  "category": "Kidney Function",
  "unit": "mg/dL",
  "range": {
   "male": [
    0.74,
    1.35
   ],
   "female": [
    0.59,
    1.04
   ],
   "general": [
    0.59,
    1.35
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A waste product from muscles that the kidneys filter out of the blood.",
  "low": "Low values can reflect low muscle mass and are usually not a concern.",
  "high": "High values can mean the kidneys are filtering less well, but dehydration, intense exercise, high-protein meals and some medicines can also raise it.",
  "why": "A key marker of how well the kidneys filter blood. It is used to estimate eGFR.",
  "prep": "Avoid very hard exercise and a very large meat meal before the test, drink normal amounts of water, and tell your doctor if you take creatine supplements.",
  "related": [
   "egfr",
   "urea",
   "bun",
   "uric_acid"
  ],
  "measures_ar": "مادة ناتجة عن العضلات تتخلص منها الكلى.",
  "why_ar": "مؤشر أساسي لكفاءة الكلى في تصفية الدم، ويُستخدم لحساب معدل الترشيح eGFR.",
  "prep_ar": "تجنب المجهود العنيف جداً والوجبات الكبيرة من اللحوم قبل التحليل، واشرب الماء بشكل معتاد، وأخبر طبيبك إن كنت تتناول مكملات الكرياتين."
 },
 {
  "key": "urea",
  "name": "Urea",
  "aliases": [
   "urea",
   "blood urea",
   "serum urea",
   "urea level"
  ],
  "category": "Kidney Function",
  "unit": "mg/dL",
  "range": {
   "general": [
    15.0,
    45.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A waste product from protein breakdown that the kidneys remove.",
  "low": "Low values can be seen with low protein intake, liver disease or overhydration.",
  "high": "High values can be linked to dehydration, high protein intake, reduced kidney function or some medicines.",
  "why": "A waste product made from protein. It is read with creatinine to look at kidney function and hydration.",
  "prep": "Dehydration and a high-protein diet can raise it. Very low protein intake or liver problems can lower it.",
  "related": [
   "creatinine",
   "bun",
   "egfr"
  ],
  "measures_ar": "ناتج من هدم البروتين يتخلص منه الجسم عن طريق الكلى.",
  "why_ar": "يُقرأ مع الكرياتينين لتقييم وظائف الكلى ومستوى الترطيب.",
  "prep_ar": "الجفاف والغذاء الغني بالبروتين قد يرفعانه، وقلة البروتين أو مشاكل الكبد قد تخفضه."
 },
 {
  "key": "bun",
  "name": "Blood Urea Nitrogen (BUN)",
  "aliases": [
   "bun",
   "blood urea nitrogen",
   "urea nitrogen"
  ],
  "category": "Kidney Function",
  "unit": "mg/dL",
  "range": {
   "general": [
    6.0,
    24.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The nitrogen part of urea, a waste product from protein breakdown that the kidneys remove.",
  "low": "Low values can be seen with low protein intake, liver disease or overhydration.",
  "high": "High values can be linked to dehydration, high protein intake, reduced kidney function or some medicines.",
  "why": "The same information as urea, reported as urea nitrogen. It is read with creatinine to look at kidney function and hydration.",
  "prep": "Dehydration and a high-protein diet can raise it.",
  "related": [
   "creatinine",
   "urea",
   "egfr"
  ],
  "measures_ar": "كمية النيتروجين في اليوريا، وهي ناتج من هدم البروتين.",
  "why_ar": "يعطي نفس معلومات اليوريا ويُقرأ مع الكرياتينين لتقييم الكلى ومستوى الترطيب.",
  "prep_ar": "الجفاف والغذاء الغني بالبروتين قد يرفعانه."
 },
 {
  "key": "uric_acid",
  "name": "Uric Acid",
  "aliases": [
   "uric acid",
   "serum uric acid",
   "urate",
   "ua"
  ],
  "category": "Kidney Function",
  "unit": "mg/dL",
  "range": {
   "male": [
    3.4,
    7.0
   ],
   "female": [
    2.4,
    6.0
   ],
   "general": [
    2.4,
    7.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A waste product from the breakdown of purines found in food and body cells.",
  "low": "Low values are usually not a concern.",
  "high": "High values can be linked to diet, alcohol, excess weight, some medicines or reduced kidney function, and can lead to gout or kidney stones in some people.",
  "why": "Looks at the risk of gout and kidney stones and helps follow kidney function.",
  "prep": "Fasting may be requested. Alcohol, sugary drinks, crash diets and dehydration can raise it.",
  "related": [
   "creatinine",
   "egfr"
  ],
  "measures_ar": "ناتج من هدم مواد تسمى البيورينات، وتتخلص منه الكلى.",
  "why_ar": "يفيد في تقييم خطر النقرس وحصوات الكلى ومتابعة وظائف الكلى.",
  "prep_ar": "قد يُطلب الصيام. الكحول والمشروبات المحلاة والحميات القاسية والجفاف قد ترفعه."
 },
 {
  "key": "egfr",
  "name": "eGFR",
  "aliases": [
   "egfr",
   "estimated gfr",
   "gfr",
   "estimated glomerular filtration rate"
  ],
  "category": "Kidney Function",
  "unit": "mL/min/1.73m2",
  "range": {
   "general": [
    60.0,
    null
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "An estimate of how well your kidneys filter the blood.",
  "low": "Values below 60 for three months or more can indicate reduced kidney function and need medical follow-up. A single result can be affected by hydration and illness.",
  "high": "High values are usually not a concern.",
  "why": "Estimates how well the kidneys filter blood and is used to follow kidney health over time.",
  "prep": "It is calculated from creatinine, age and sex, so the preparation is the same as for creatinine. The trend over time matters more than one value.",
  "related": [
   "creatinine",
   "urea"
  ],
  "measures_ar": "تقدير لمعدل ترشيح الكلى للدم في الدقيقة.",
  "why_ar": "يقدّر كفاءة الكلى في تصفية الدم ويُستخدم لمتابعة صحة الكلى عبر الزمن.",
  "prep_ar": "يُحسب من الكرياتينين والعمر والجنس، لذلك تجهيزه مثل الكرياتينين. اتجاه النتائج عبر الزمن أهم من قيمة واحدة."
 },
 {
  "key": "sodium",
  "name": "Sodium",
  "aliases": [
   "sodium",
   "na",
   "serum sodium",
   "sodium na"
  ],
  "category": "Electrolytes and Minerals",
  "unit": "mmol/L",
  "range": {
   "general": [
    135.0,
    145.0
   ]
  },
  "critical": {
   "low": 120.0,
   "high": 160.0
  },
  "measures": "A mineral that helps control body water and supports nerve and muscle function.",
  "low": "Low sodium can be caused by drinking too much water, some medicines, or heart, kidney or liver conditions. Severe low sodium needs urgent care.",
  "high": "High sodium is most often caused by not drinking enough fluids.",
  "why": "Shows water and salt balance, which matters for nerves, muscles and blood pressure.",
  "prep": "No fasting needed. Heavy sweating, vomiting, diarrhea and the amount of fluid you drink can change it.",
  "related": [
   "potassium",
   "chloride"
  ],
  "measures_ar": "مستوى الصوديوم، وهو ملح أساسي ينظم توازن الماء في الجسم.",
  "why_ar": "يوضح توازن الماء والأملاح، وهو مهم للأعصاب والعضلات وضغط الدم.",
  "prep_ar": "لا يحتاج إلى صيام. التعرق الشديد والقيء والإسهال وكمية السوائل التي تشربها قد تغيّره."
 },
 {
  "key": "potassium",
  "name": "Potassium",
  "aliases": [
   "potassium",
   "k",
   "serum potassium",
   "potassium k"
  ],
  "category": "Electrolytes and Minerals",
  "unit": "mmol/L",
  "range": {
   "general": [
    3.5,
    5.0
   ]
  },
  "critical": {
   "low": 2.5,
   "high": 6.5
  },
  "measures": "A mineral that is essential for heart rhythm and muscle function.",
  "low": "Low values can follow vomiting, diarrhea, water pills or low intake, and can affect heart rhythm if severe.",
  "high": "High values can be linked to kidney problems, some medicines, or a blood sample that was damaged in the lab. Very high levels can affect heart rhythm.",
  "why": "Essential for heart rhythm and muscle function. Both low and high levels can matter.",
  "prep": "A clenched fist, a delay before the sample is processed or a damaged sample can falsely raise it. If the value is unexpected, ask whether to repeat it.",
  "related": [
   "sodium",
   "creatinine",
   "magnesium"
  ],
  "measures_ar": "مستوى البوتاسيوم، وهو معدن مهم لانتظام القلب وعمل العضلات.",
  "why_ar": "ضروري لانتظام ضربات القلب ووظيفة العضلات، وقد يكون الانخفاض أو الارتفاع مهماً.",
  "prep_ar": "قبض اليد بقوة أو تأخر معالجة العينة أو تلفها قد يرفعه بشكل كاذب. إذا كانت النتيجة غير متوقعة فاسأل عن إعادة التحليل."
 },
 {
  "key": "chloride",
  "name": "Chloride",
  "aliases": [
   "chloride",
   "cl",
   "serum chloride"
  ],
  "category": "Electrolytes and Minerals",
  "unit": "mmol/L",
  "range": {
   "general": [
    98.0,
    107.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A mineral that works with sodium to balance body fluids and acid levels.",
  "low": "Low values can follow vomiting or dehydration and are interpreted together with sodium and other tests.",
  "high": "High values can be linked to dehydration or to acid balance changes and are interpreted together with sodium and other tests.",
  "why": "Works with sodium to show fluid and acid-base balance.",
  "prep": "No fasting needed. Vomiting, diarrhea and heavy sweating can change it.",
  "related": [
   "sodium",
   "potassium"
  ],
  "measures_ar": "مستوى الكلوريد، وهو معدن يعمل مع الصوديوم لتوازن السوائل والحموضة.",
  "why_ar": "يعمل مع الصوديوم لتوضيح توازن السوائل والحموضة في الجسم.",
  "prep_ar": "لا يحتاج إلى صيام. القيء والإسهال والتعرق الشديد قد تغيّره."
 },
 {
  "key": "calcium",
  "name": "Calcium",
  "aliases": [
   "calcium",
   "total calcium",
   "serum calcium",
   "calcium total",
   "ca"
  ],
  "category": "Electrolytes and Minerals",
  "unit": "mg/dL",
  "range": {
   "general": [
    8.5,
    10.2
   ]
  },
  "critical": {
   "low": 6.5,
   "high": 13.0
  },
  "measures": "A mineral that is important for bones, nerves, muscles and the heart.",
  "low": "Low values can be linked to vitamin D deficiency, low albumin, or kidney or parathyroid problems.",
  "high": "High values can be linked to an overactive parathyroid gland, too much vitamin D or calcium supplements, or some other conditions.",
  "why": "Checks bones, nerves, muscles and the parathyroid glands.",
  "prep": "No special preparation. Tell your doctor about calcium or vitamin D supplements. Albumin is often checked with it.",
  "related": [
   "albumin",
   "vitamin_d",
   "phosphorus",
   "magnesium"
  ],
  "measures_ar": "مستوى الكالسيوم في الدم، وهو مهم للعظام والأعصاب والعضلات.",
  "why_ar": "يفحص العظام والأعصاب والعضلات ووظيفة الغدد جارات الدرقية.",
  "prep_ar": "لا يحتاج إلى تجهيز خاص. أخبر طبيبك بمكملات الكالسيوم أو فيتامين د، وغالباً يُفحص الألبومين معه."
 },
 {
  "key": "magnesium",
  "name": "Magnesium",
  "aliases": [
   "magnesium",
   "serum magnesium",
   "magnesium mg"
  ],
  "category": "Electrolytes and Minerals",
  "unit": "mg/dL",
  "range": {
   "general": [
    1.7,
    2.2
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A mineral that supports muscles, nerves and heart rhythm.",
  "low": "Low values can be linked to poor intake, diarrhea, alcohol use or some medicines.",
  "high": "High values are uncommon and are usually linked to kidney problems or magnesium-containing medicines.",
  "why": "Supports muscles, nerves and heart rhythm. Low levels often go along with low potassium or calcium.",
  "prep": "No fasting needed. Some long-term medicines and heavy alcohol use can lower it, so tell your doctor what you take.",
  "related": [
   "calcium",
   "potassium"
  ],
  "measures_ar": "مستوى المغنيسيوم، وهو معدن مهم للعضلات والأعصاب وانتظام القلب.",
  "why_ar": "يدعم العضلات والأعصاب وانتظام القلب، وانخفاضه كثيراً ما يرافق انخفاض البوتاسيوم أو الكالسيوم.",
  "prep_ar": "لا يحتاج إلى صيام. بعض الأدوية طويلة الأمد والإفراط في الكحول قد يخفضانه، لذلك أخبر طبيبك بما تتناوله."
 },
 {
  "key": "phosphorus",
  "name": "Phosphorus",
  "aliases": [
   "phosphorus",
   "phosphate",
   "serum phosphorus",
   "inorganic phosphorus"
  ],
  "category": "Electrolytes and Minerals",
  "unit": "mg/dL",
  "range": {
   "general": [
    2.5,
    4.5
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A mineral that works with calcium to build bones and supports energy production.",
  "low": "Low values can be linked to poor intake, vitamin D deficiency or some medicines.",
  "high": "High values can be linked to reduced kidney function or to parathyroid and vitamin D imbalances.",
  "why": "Works with calcium for bone health and helps assess kidney and parathyroid conditions.",
  "prep": "Fasting is often preferred because meals can lower it.",
  "related": [
   "calcium",
   "vitamin_d",
   "creatinine"
  ],
  "measures_ar": "مستوى الفوسفور، وهو معدن يعمل مع الكالسيوم لبناء العظام.",
  "why_ar": "يعمل مع الكالسيوم لصحة العظام ويساعد على تقييم حالات الكلى والغدد جارات الدرقية.",
  "prep_ar": "يُفضل غالباً الصيام لأن الوجبات قد تخفضه."
 },
 {
  "key": "tsh",
  "name": "TSH",
  "aliases": [
   "tsh",
   "thyroid stimulating hormone",
   "thyrotropin",
   "tsh level"
  ],
  "category": "Thyroid",
  "unit": "mIU/L",
  "range": {
   "general": [
    0.4,
    4.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A hormone from the pituitary gland that tells the thyroid how much thyroid hormone to make.",
  "low": "A low TSH can point to an overactive thyroid or to taking too much thyroid medication.",
  "high": "A high TSH can point to an underactive thyroid. Your doctor will usually look at free T4 as well.",
  "why": "The first test usually used to screen for an underactive or overactive thyroid.",
  "prep": "No fasting needed, but time of day matters because TSH is higher at night and early morning. Biotin supplements can interfere, so tell your doctor if you take them.",
  "related": [
   "free_t4",
   "free_t3"
  ],
  "measures_ar": "هرمون تفرزه الغدة النخامية ليتحكم في نشاط الغدة الدرقية.",
  "why_ar": "غالباً هو أول تحليل للكشف عن كسل الغدة الدرقية أو نشاطها الزائد.",
  "prep_ar": "لا يحتاج إلى صيام، لكن وقت اليوم مهم لأنه يرتفع ليلاً وفي الصباح الباكر. مكملات البيوتين قد تتداخل مع التحليل، فأخبر طبيبك إن كنت تتناولها."
 },
 {
  "key": "free_t4",
  "name": "Free T4",
  "aliases": [
   "free t4",
   "ft4",
   "free thyroxine",
   "t4 free",
   "thyroxine free"
  ],
  "category": "Thyroid",
  "unit": "ng/dL",
  "range": {
   "general": [
    0.8,
    1.8
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The active thyroid hormone circulating in the blood.",
  "low": "Low values together with a high TSH suggest an underactive thyroid.",
  "high": "High values together with a low TSH suggest an overactive thyroid.",
  "why": "Shows the level of the active thyroid hormone. It is interpreted together with TSH.",
  "prep": "Same preparation as TSH. Biotin supplements can interfere.",
  "related": [
   "tsh",
   "free_t3"
  ],
  "measures_ar": "المستوى الحر النشط من هرمون الغدة الدرقية (T4).",
  "why_ar": "يوضح مستوى هرمون الغدة الدرقية النشط، ويُفسَّر مع TSH.",
  "prep_ar": "نفس تجهيز TSH. مكملات البيوتين قد تتداخل مع التحليل."
 },
 {
  "key": "free_t3",
  "name": "Free T3",
  "aliases": [
   "free t3",
   "ft3",
   "free triiodothyronine",
   "t3 free"
  ],
  "category": "Thyroid",
  "unit": "pg/mL",
  "range": {
   "general": [
    2.3,
    4.2
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The most active form of thyroid hormone in the blood.",
  "low": "Low values can be seen with an underactive thyroid or during illness.",
  "high": "High values can be seen with an overactive thyroid.",
  "why": "Shows the most active thyroid hormone. It helps when an overactive thyroid is suspected.",
  "prep": "Same preparation as TSH. Biotin supplements can interfere.",
  "related": [
   "tsh",
   "free_t4"
  ],
  "measures_ar": "المستوى الحر من الشكل الأنشط لهرمون الغدة الدرقية (T3).",
  "why_ar": "يوضح أنشط أشكال هرمون الغدة الدرقية، ويفيد عند الاشتباه في نشاط زائد.",
  "prep_ar": "نفس تجهيز TSH. مكملات البيوتين قد تتداخل مع التحليل."
 },
 {
  "key": "vitamin_d",
  "name": "Vitamin D (25-OH)",
  "aliases": [
   "vitamin d",
   "vitamin d 25oh",
   "25oh vitamin d",
   "25oh d",
   "vit d",
   "vitamin d3",
   "25oh vitamin d3",
   "vitamin d total"
  ],
  "category": "Vitamins and Iron",
  "unit": "ng/mL",
  "range": {
   "general": [
    30.0,
    100.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The main stored form of vitamin D, which is important for bones and calcium absorption.",
  "low": "Below 20 ng/mL is generally considered deficient and 20 to 29 ng/mL insufficient. Common causes are limited sun exposure, covered skin, darker skin, low dietary intake, and some medical conditions.",
  "high": "Very high values usually come from taking high-dose supplements for a long time.",
  "why": "Vitamin D supports bones and muscles. Low levels are common, especially with little sun exposure.",
  "prep": "No fasting needed. The season, recent sun exposure and recent supplements can affect it.",
  "related": [
   "calcium",
   "phosphorus",
   "alp"
  ],
  "measures_ar": "مستوى فيتامين د في الدم، وهو يدعم امتصاص الكالسيوم وصحة العظام.",
  "why_ar": "يدعم العظام والعضلات، وانخفاضه شائع خاصة مع قلة التعرض للشمس.",
  "prep_ar": "لا يحتاج إلى صيام. الموسم والتعرض الحديث للشمس والمكملات الحديثة قد تؤثر عليه."
 },
 {
  "key": "vitamin_b12",
  "name": "Vitamin B12",
  "aliases": [
   "vitamin b12",
   "b12",
   "vit b12",
   "cobalamin",
   "serum b12",
   "vitamin b 12"
  ],
  "category": "Vitamins and Iron",
  "unit": "pg/mL",
  "range": {
   "general": [
    200.0,
    900.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A vitamin needed for healthy nerves and for making red blood cells.",
  "low": "Low values can come from low dietary intake, absorption problems, some stomach conditions, or long-term use of certain medicines such as metformin or acid reducers.",
  "high": "High values are often due to supplements and are usually not a concern.",
  "why": "Checks one cause of anemia and of nerve symptoms such as tingling or numbness.",
  "prep": "Fasting may be requested. Tell your doctor if you follow a vegetarian or vegan diet or take long-term medicines, since some can lower B12.",
  "related": [
   "folate",
   "hemoglobin",
   "mcv"
  ],
  "measures_ar": "مستوى فيتامين B12 اللازم لتكوين كرات الدم وسلامة الأعصاب.",
  "why_ar": "يفحص أحد أسباب الأنيميا ومشاكل الأعصاب مثل التنميل والخدر.",
  "prep_ar": "قد يُطلب الصيام. أخبر طبيبك إن كنت نباتياً أو تتناول أدوية طويلة الأمد لأن بعضها قد يخفض B12."
 },
 {
  "key": "folate",
  "name": "Folate",
  "aliases": [
   "folate",
   "folic acid",
   "serum folate",
   "vitamin b9"
  ],
  "category": "Vitamins and Iron",
  "unit": "ng/mL",
  "range": {
   "general": [
    4.0,
    20.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A vitamin needed to make red blood cells and new cells in general.",
  "low": "Low values are linked to low dietary intake, alcohol use or absorption problems, and can cause anemia.",
  "high": "High values are usually from supplements.",
  "why": "Checks another cause of anemia. It is also important when planning a pregnancy.",
  "prep": "Fasting may be requested because a recent meal can raise it.",
  "related": [
   "vitamin_b12",
   "mcv",
   "hemoglobin"
  ],
  "measures_ar": "مستوى حمض الفوليك (فيتامين B9) اللازم لتكوين كرات الدم.",
  "why_ar": "يفحص سبباً آخر للأنيميا، وهو مهم أيضاً عند التخطيط للحمل.",
  "prep_ar": "قد يُطلب الصيام لأن الوجبة الأخيرة قد ترفعه."
 },
 {
  "key": "ferritin",
  "name": "Ferritin",
  "aliases": [
   "ferritin",
   "serum ferritin"
  ],
  "category": "Vitamins and Iron",
  "unit": "ng/mL",
  "range": {
   "male": [
    24.0,
    336.0
   ],
   "female": [
    11.0,
    307.0
   ],
   "general": [
    11.0,
    336.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A protein that stores iron, which reflects your body's iron reserves.",
  "low": "Low ferritin is the earliest sign of iron deficiency. It is often caused by blood loss, including heavy periods, low dietary iron or poor absorption.",
  "high": "High ferritin can be linked to inflammation, liver conditions, infection or iron overload, because ferritin also rises as a general inflammation marker.",
  "why": "The best single test of your body's iron stores. It can fall before anemia appears.",
  "prep": "Fasting is often preferred. Ferritin also rises with inflammation, infection and liver problems, so a normal value can hide low iron when there is inflammation.",
  "related": [
   "iron",
   "hemoglobin",
   "mcv",
   "crp"
  ],
  "measures_ar": "بروتين يخزّن الحديد في الجسم، ومستواه يعكس مخزون الحديد.",
  "why_ar": "أفضل تحليل منفرد لمخزون الحديد، وقد ينخفض قبل ظهور الأنيميا.",
  "prep_ar": "يُفضل غالباً الصيام. يرتفع الفيريتين مع الالتهاب والعدوى ومشاكل الكبد، فقد تبدو النتيجة طبيعية رغم نقص الحديد."
 },
 {
  "key": "iron",
  "name": "Serum Iron",
  "aliases": [
   "iron",
   "serum iron",
   "iron serum",
   "fe"
  ],
  "category": "Vitamins and Iron",
  "unit": "ug/dL",
  "range": {
   "general": [
    60.0,
    170.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "The amount of iron circulating in your blood.",
  "low": "Low values can be linked to iron deficiency, blood loss or poor absorption.",
  "high": "High values can come from iron supplements or iron overload conditions, and the level varies a lot during the day.",
  "why": "Measures the iron currently circulating in your blood. It is read together with ferritin.",
  "prep": "Usually a morning sample after an overnight fast. Iron supplements taken in the last day or two can falsely raise it, so tell your doctor.",
  "related": [
   "ferritin",
   "hemoglobin"
  ],
  "measures_ar": "كمية الحديد المتداولة في الدم حالياً.",
  "why_ar": "يقيس الحديد الموجود في الدم وقت التحليل، ويُقرأ مع الفيريتين.",
  "prep_ar": "غالباً عينة صباحية بعد صيام الليل. مكملات الحديد خلال اليوم أو اليومين السابقين قد ترفعه بشكل كاذب، فأخبر طبيبك."
 },
 {
  "key": "crp",
  "name": "C-Reactive Protein (CRP)",
  "aliases": [
   "crp",
   "c reactive protein",
   "c reactive protein crp"
  ],
  "category": "Inflammation and Clotting",
  "unit": "mg/L",
  "range": {
   "general": [
    0.0,
    10.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A marker that rises with inflammation or infection.",
  "low": "Low values are normal.",
  "high": "High values suggest inflammation or infection somewhere in the body but do not show where. Your doctor will interpret them together with your symptoms.",
  "why": "A quick marker of inflammation or infection.",
  "prep": "No fasting needed. It is not specific, so many conditions can raise it.",
  "related": [
   "esr",
   "wbc",
   "hs_crp"
  ],
  "measures_ar": "بروتين يرتفع في الدم عند وجود التهاب أو عدوى.",
  "why_ar": "مؤشر سريع على وجود التهاب أو عدوى.",
  "prep_ar": "لا يحتاج إلى صيام. هو غير نوعي، فكثير من الحالات قد ترفعه."
 },
 {
  "key": "hs_crp",
  "name": "High-Sensitivity CRP",
  "aliases": [
   "hs crp",
   "high sensitivity crp",
   "hscrp",
   "high sensitivity c reactive protein"
  ],
  "category": "Inflammation and Clotting",
  "unit": "mg/L",
  "range": {
   "general": [
    0.0,
    3.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "A very sensitive CRP test used to assess low-level inflammation and heart risk.",
  "low": "Low values are normal.",
  "high": "Values above 3 mg/L suggest a higher heart risk, and values above 10 mg/L can reflect an active infection or inflammation.",
  "why": "A more sensitive CRP test used to look at low-grade inflammation linked with heart risk.",
  "prep": "Best measured when you are well. A cold, an injury or an infection can raise it, and doctors may repeat the test.",
  "related": [
   "crp",
   "ldl",
   "total_cholesterol"
  ],
  "measures_ar": "نسخة أكثر حساسية من CRP تكشف الالتهاب الخفيف.",
  "why_ar": "يُستخدم لتقدير الالتهاب الخفيف المرتبط بمخاطر القلب.",
  "prep_ar": "الأفضل قياسه وأنت بصحة جيدة. نزلة البرد أو الإصابة أو العدوى قد ترفعه، وقد يعيد الطبيب التحليل."
 },
 {
  "key": "esr",
  "name": "ESR",
  "aliases": [
   "esr",
   "erythrocyte sedimentation rate",
   "sed rate",
   "esr 1st hour"
  ],
  "category": "Inflammation and Clotting",
  "unit": "mm/hr",
  "range": {
   "male": [
    0.0,
    15.0
   ],
   "female": [
    0.0,
    20.0
   ],
   "general": [
    0.0,
    20.0
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "How fast red blood cells settle in a tube, which is a general sign of inflammation.",
  "low": "Low values are usually not a concern.",
  "high": "High values are non-specific and can be linked to infection, inflammation, anemia or pregnancy.",
  "why": "A slow general marker of inflammation that helps follow some conditions over time.",
  "prep": "No fasting needed. Anemia, pregnancy and older age raise it.",
  "related": [
   "crp",
   "wbc"
  ],
  "measures_ar": "سرعة ترسّب كرات الدم الحمراء في أنبوب خلال ساعة، وهي مؤشر عام للالتهاب.",
  "why_ar": "مؤشر عام وبطيء للالتهاب يساعد على متابعة بعض الحالات عبر الزمن.",
  "prep_ar": "لا يحتاج إلى صيام. الأنيميا والحمل وكبر السن ترفعه."
 },
 {
  "key": "inr",
  "name": "INR",
  "aliases": [
   "inr",
   "international normalized ratio",
   "pt inr"
  ],
  "category": "Inflammation and Clotting",
  "unit": "ratio",
  "range": {
   "general": [
    0.8,
    1.1
   ]
  },
  "critical": {
   "low": null,
   "high": null
  },
  "measures": "How long your blood takes to clot compared with a standard.",
  "low": "Low values are generally not a concern.",
  "high": "High values mean the blood takes longer to clot. This is expected if you take blood thinners such as warfarin, and can also be linked to liver disease or vitamin K deficiency.",
  "why": "Measures how long your blood takes to clot. Used to assess liver function and to monitor blood-thinning treatment.",
  "prep": "Follow the schedule your doctor gave you. Foods rich in vitamin K and some medicines affect it, so tell your doctor what you take.",
  "related": [
   "platelets",
   "alt"
  ],
  "measures_ar": "مقياس قياسي للوقت الذي يستغرقه الدم ليتجلط.",
  "why_ar": "يقيس زمن تجلط الدم، ويُستخدم لتقييم وظائف الكبد ومتابعة العلاج المسيّل للدم.",
  "prep_ar": "اتبع الجدول الذي حدده طبيبك. الأطعمة الغنية بفيتامين K وبعض الأدوية تؤثر عليه، فأخبر طبيبك بما تتناوله."
 }
]

Writing medical_explainer/knowledge_base.json


## 4) Core logic
Extraction rules, unit handling, range comparison, % distance from the range and the pattern rules. No model is used here.


In [4]:
%%writefile medical_explainer/medical_core.py
import json
import re
from difflib import get_close_matches

FLAG_TOKENS = {"h", "l", "hh", "ll", "n", "a", "high", "low", "normal", "h*", "l*"}

UNIT_MAP = {
    "g/dl": "g/dl", "gm/dl": "g/dl", "g/100ml": "g/dl", "gm%": "g/dl",
    "mg/dl": "mg/dl",
    "mmol/l": "mmol/l", "meq/l": "mmol/l",
    "%": "%",
    "fl": "fl",
    "pg": "pg",
    "x103/ul": "10^3/ul", "103/ul": "10^3/ul", "x103/mm3": "10^3/ul", "103/mm3": "10^3/ul",
    "k/ul": "10^3/ul", "x109/l": "10^3/ul", "109/l": "10^3/ul", "x10e3/ul": "10^3/ul",
    "10e3/ul": "10^3/ul", "x103/cmm": "10^3/ul", "103/cmm": "10^3/ul", "thou/ul": "10^3/ul",
    "x106/ul": "10^6/ul", "106/ul": "10^6/ul", "x106/mm3": "10^6/ul", "106/mm3": "10^6/ul",
    "m/ul": "10^6/ul", "x1012/l": "10^6/ul", "1012/l": "10^6/ul", "million/ul": "10^6/ul",
    "mil/ul": "10^6/ul", "x106/cmm": "10^6/ul", "106/cmm": "10^6/ul",
    "cumm": "cells/ul", "/cumm": "cells/ul", "/ul": "cells/ul", "/mm3": "cells/ul",
    "cells/ul": "cells/ul", "cells/mm3": "cells/ul", "cmm": "cells/ul", "/cmm": "cells/ul",
    "u/l": "u/l", "iu/l": "u/l", "ui/l": "u/l",
    "ng/ml": "ng/ml", "ug/l": "ng/ml",
    "pg/ml": "pg/ml",
    "ng/dl": "ng/dl",
    "ug/dl": "ug/dl", "mcg/dl": "ug/dl",
    "mg/l": "mg/l",
    "miu/l": "miu/l", "uiu/ml": "miu/l", "muiu/ml": "miu/l",
    "mm/hr": "mm/hr", "mm/h": "mm/hr", "mm/1sthr": "mm/hr", "mm/hour": "mm/hr", "mm/1hr": "mm/hr",
    "ml/min/173m2": "egfr", "ml/min": "egfr",
    "ratio": "ratio",
    "pmol/l": "pmol/l", "nmol/l": "nmol/l", "umol/l": "umol/l",
}

UNIT_PATTERN = (
    r"(?:[xX\u00d7]\s*)?10\s*\^?\s*[0-9\u00b3\u2076\u2079]+\s*/\s*[A-Za-z\u00b5\u03bc]+"
    r"|[A-Za-z\u00b5\u03bc%/][A-Za-z\u00b5\u03bc%0-9\^/\.]*"
)

LINE_PATTERN = re.compile(
    r"^\s*(?P<name>[A-Za-z][A-Za-z0-9 ,()\-/'.%]*?)"
    r"(?:\s*[:|]\s*|\s+)"
    r"(?P<value>\d+(?:[.,]\d+)?)(?![A-Za-z0-9\-])"
    r"(?P<tail>.*)$"
)

UNIT_AT_START = re.compile(r"^\s*(?P<unit>" + UNIT_PATTERN + r")")


def normalize_name(text):
    text = str(text).lower()
    text = re.sub(r"25\s*-?\s*(hydroxy|oh)", "25oh", text)
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.strip()


def canonical_unit(unit):
    if not unit:
        return None
    value = str(unit).lower()
    replacements = (
        ("\u00b5", "u"), ("\u03bc", "u"), ("\u00d7", "x"),
        ("\u00b3", "3"), ("\u2076", "6"), ("\u2079", "9"), ("\u00b2", "2"),
    )
    for old, new in replacements:
        value = value.replace(old, new)
    value = re.sub(r"[\s\^\*]", "", value)
    value = value.replace("1.73m2", "173m2")
    value = value.rstrip(".")
    return UNIT_MAP.get(value)


def to_float(value):
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    match = re.search(r"\d+(?:[.,]\d+)*", str(value))
    if not match:
        return None
    token = match.group(0)
    if re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", token):
        token = token.replace(",", "")
    else:
        token = token.replace(",", ".")
    try:
        return float(token)
    except ValueError:
        return None


def parse_range_text(text):
    if not text:
        return None
    value = str(text).replace("\u2013", "-").replace("\u2014", "-")
    number = r"(\d+(?:[.,]\d+)?)"
    between = re.findall(number + r"\s*(?:-|to)\s*" + number, value, flags=re.I)
    if len(between) == 1:
        low = to_float(between[0][0])
        high = to_float(between[0][1])
        if low is not None and high is not None and low <= high:
            return (low, high)
        return None
    if len(between) > 1:
        return None
    upper = re.search(r"(?:<=|\u2264|<|up to|upto|less than|below|under)\s*" + number, value, flags=re.I)
    if upper:
        return (None, to_float(upper.group(1)))
    lower = re.search(r"(?:>=|\u2265|>|more than|above|greater than|over)\s*" + number, value, flags=re.I)
    if lower:
        return (to_float(lower.group(1)), None)
    return None


def classify(value, low, high):
    if low is not None and value < low:
        return "Low"
    if high is not None and value > high:
        return "High"
    return "Normal"


def format_range(low, high):
    if low is not None and high is not None:
        return f"{low:g} - {high:g}"
    if high is not None:
        return f"<= {high:g}"
    if low is not None:
        return f">= {low:g}"
    return ""


def load_knowledge_base(path):
    with open(path, "r", encoding="utf-8") as handle:
        entries = json.load(handle)
    alias_index = {}
    by_key = {}
    for entry in entries:
        by_key[entry["key"]] = entry
        names = set(entry.get("aliases", []))
        names.add(entry["name"])
        names.add(entry["key"].replace("_", " "))
        for alias in names:
            normalized = normalize_name(alias)
            if normalized:
                alias_index[normalized] = entry
    return {
        "entries": entries,
        "by_key": by_key,
        "alias_index": alias_index,
        "alias_keys": list(alias_index.keys()),
    }


def name_candidates(name):
    raw = str(name)
    base = re.sub(r"\(.*?\)", " ", raw)
    inner = re.findall(r"\((.*?)\)", raw)
    candidates = [normalize_name(base), normalize_name(raw)] + [normalize_name(item) for item in inner]
    unique = []
    for candidate in candidates:
        if candidate and candidate not in unique:
            unique.append(candidate)
    return unique


def find_entry(name, kb):
    index = kb["alias_index"]
    candidates = name_candidates(name)
    for candidate in candidates:
        if candidate in index:
            return index[candidate]
    for candidate in candidates:
        if " " in candidate or len(candidate) < 6:
            continue
        pool = [key for key in kb["alias_keys"] if " " not in key and len(key) >= 6]
        close = get_close_matches(candidate, pool, n=1, cutoff=0.9)
        if close:
            return index[close[0]]
    return None


def kb_range(entry, sex):
    ranges = entry.get("range", {})
    chosen = ranges.get(sex) if sex in ("male", "female") else None
    if not chosen:
        chosen = ranges.get("general")
    if not chosen:
        return None, None
    return chosen[0], chosen[1]


def is_urgent(value, entry):
    if not entry:
        return False
    critical = entry.get("critical") or {}
    low = critical.get("low")
    high = critical.get("high")
    if low is not None and value <= low:
        return True
    if high is not None and value >= high:
        return True
    return False


def evaluate_tests(items, kb, sex=None):
    rows = []
    seen = set()
    for item in items:
        name = str(item.get("name", "")).strip()
        raw_value = item.get("value")
        if isinstance(raw_value, str) and raw_value.strip()[:1] in ("<", ">"):
            continue
        value = to_float(raw_value)
        if not name or value is None:
            continue
        signature = (normalize_name(name), value)
        if signature in seen:
            continue
        seen.add(signature)

        unit = str(item.get("unit") or "").strip()
        range_text = str(item.get("reference_range") or "").strip()
        entry = find_entry(name, kb)
        report_range = parse_range_text(range_text)

        low = None
        high = None
        source = ""
        note = ""
        unit_ok = True

        if entry:
            kb_unit = canonical_unit(entry.get("unit"))
            report_unit = canonical_unit(unit)
            if kb_unit and report_unit and kb_unit != report_unit:
                unit_ok = False

        if report_range:
            low, high = report_range
            source = "report"
        elif entry and unit_ok:
            low, high = kb_range(entry, sex)
            source = "reference table"
            if high is not None and high > 0 and value > high * 200:
                unit_ok = False
                low = None
                high = None
                source = ""
        if entry and not unit_ok and not report_range:
            note = "The unit in the report differs from the reference table, so this result was not compared. Use the range printed in your report."
        if entry and entry.get("note") and source == "reference table":
            note = entry["note"]

        status = "Unknown"
        if low is not None or high is not None:
            status = classify(value, low, high)

        scale_ok = False
        if entry and unit_ok:
            kb_unit = canonical_unit(entry.get("unit"))
            report_unit = canonical_unit(unit)
            if report_unit and kb_unit == report_unit:
                scale_ok = True
            elif not unit:
                _, kb_high = kb_range(entry, sex)
                scale_ok = kb_high is None or value <= kb_high * 200
        urgent = scale_ok and is_urgent(value, entry)

        direction, pct = deviation(value, low, high)
        rows.append({
            "direction": direction,
            "deviation_pct": pct,
            "severity": severity_of(pct),
            "input_name": name,
            "name": entry["name"] if entry else name,
            "key": entry["key"] if entry else None,
            "category": entry["category"] if entry else "Other",
            "value": value,
            "unit": unit or (entry.get("unit", "") if entry else ""),
            "range": format_range(low, high),
            "range_source": source,
            "status": status,
            "urgent": urgent,
            "note": note,
            "measures": entry["measures"] if entry else "",
            "in_knowledge_base": bool(entry),
            "why": entry.get("why", "") if entry else "",
            "prep": entry.get("prep", "") if entry else "",
            "measures_ar": entry.get("measures_ar", "") if entry else "",
            "why_ar": entry.get("why_ar", "") if entry else "",
            "prep_ar": entry.get("prep_ar", "") if entry else "",
            "explanation": "",
            "sections": {},
        })
    return rows


def preclean_line(line):
    line = line.replace("|", "  ").replace("\t", "  ")
    line = re.sub(r"25\s*-?\s*hydroxy", "25OH", line, flags=re.I)
    return line


def regex_extract(report_text, kb):
    items = []
    for raw_line in str(report_text).splitlines():
        line = preclean_line(raw_line)
        match = LINE_PATTERN.match(line)
        if not match:
            continue
        name = match.group("name").strip(" ,:-")
        tail = match.group("tail")
        unit = ""
        unit_match = UNIT_AT_START.match(tail)
        if unit_match:
            candidate = unit_match.group("unit").strip()
            if candidate.lower() not in FLAG_TOKENS:
                unit = candidate
            tail = tail[unit_match.end():]
        range_text = ""
        parsed = parse_range_text(tail)
        if parsed:
            range_text = tail.strip()
        if not name:
            continue
        if find_entry(name, kb) is None and not parsed:
            continue
        items.append({
            "name": name,
            "value": match.group("value"),
            "unit": unit,
            "reference_range": range_text,
        })
    return items


def filter_by_text(items, report_text):
    numbers = set()
    for token in re.findall(r"\d+(?:[.,]\d+)*", str(report_text)):
        converted = to_float(token)
        if converted is not None:
            numbers.add(converted)
    kept = []
    for item in items:
        value = to_float(item.get("value"))
        if value is not None and value in numbers:
            kept.append(item)
    return kept


def extract_json_block(text):
    matches = re.findall(r"```json\s*(.*?)\s*```", text, flags=re.DOTALL)
    if matches:
        return matches[-1]
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        return text[start:end + 1]
    return text.strip()


def summarize_counts(rows):
    counts = {"total": len(rows), "normal": 0, "high": 0, "low": 0, "unknown": 0, "urgent": 0}
    for row in rows:
        counts[row["status"].lower()] += 1
        if row["urgent"]:
            counts["urgent"] += 1
    return counts


def default_questions():
    return [
        "What could be causing the results that are outside the reference range?",
        "Do I need to repeat any of these tests, and when?",
        "Could my medicines, diet or daily habits be affecting these results?",
        "Do you recommend any additional tests?",
    ]


def fallback_summary(rows):
    counts = summarize_counts(rows)
    abnormal = [row for row in rows if row["status"] in ("Low", "High")]
    if not abnormal:
        return f"All {counts['normal']} evaluated results are within their reference ranges."
    names = ", ".join(f"{row['name']} ({row['status'].lower()})" for row in abnormal[:8])
    return (
        f"{len(abnormal)} of {counts['total']} results are outside their reference ranges: {names}. "
        "Please discuss these results with your doctor."
    )


# explanation helpers
STATUS_AR = {"High": "مرتفع", "Low": "منخفض", "Normal": "طبيعي", "Unknown": "غير مُقيَّم"}
DISTANCE_WORDS = {
    "en": {"mild": "small", "moderate": "moderate", "marked": "large"},
    "ar": {"mild": "صغيرة", "moderate": "متوسطة", "marked": "كبيرة"},
}


def deviation(value, low, high):
    # how far the value is outside the range, as % of the nearest limit
    if low is not None and value < low and low:
        return "below", (low - value) / abs(low) * 100
    if high is not None and value > high and high:
        return "above", (value - high) / abs(high) * 100
    return None, None


def severity_of(pct):
    if pct is None:
        return ""
    if pct < 10:
        return "mild"
    if pct < 30:
        return "moderate"
    return "marked"


def describe_distance(row, language="English"):
    pct = row.get("deviation_pct")
    direction = row.get("direction")
    if pct is None or not direction:
        return ""
    arabic = language == "Arabic"
    word = DISTANCE_WORDS["ar" if arabic else "en"][row["severity"]]
    if arabic:
        side = "أقل من الحد الأدنى" if direction == "below" else "أعلى من الحد الأعلى"
        return f"النتيجة {side} للمدى المرجعي بنحو {pct:.0f}% (مسافة {word} عن المدى)."
    side = "below the lower limit" if direction == "below" else "above the upper limit"
    return f"The result is about {pct:.0f}% {side} of the reference range (a {word} distance from the range)."


def short_distance(row):
    pct = row.get("deviation_pct")
    if pct is None or not row.get("direction"):
        return ""
    return f"{pct:.0f}% {row['direction']}"


def related_results(row, rows, kb):
    entry = kb["by_key"].get(row["key"]) if row.get("key") else None
    if not entry:
        return []
    by_key = {r["key"]: r for r in rows if r.get("key")}
    return [by_key[key] for key in entry.get("related", []) if key in by_key and key != row["key"]]


def format_related(related):
    return "; ".join(f"{r['name']}: {r['value']:g} {r['unit']} ({r['status']})" for r in related)


SECTION_TAGS = ["MEANING", "YOUR RESULT", "COMMON REASONS", "RELATED RESULTS", "ASK YOUR DOCTOR"]
TAG_RE = re.compile(
    r"^[\s*#>\-]*\**\s*(MEANING|YOUR RESULT|COMMON REASONS|RELATED RESULTS|ASK YOUR DOCTOR)\s*\**\s*:\s*\**\s*",
    flags=re.I | re.M,
)


def parse_sections(text):
    # split the model answer by its labels, returns {} if the labels are missing
    matches = list(TAG_RE.finditer(text))
    if not matches:
        return {}
    sections = {}
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        body = text[match.end():end].strip().strip("*").strip()
        if body and body.lower().strip(". ") != "none":
            sections[match.group(1).upper()] = body
    return sections


def detect_patterns(rows):
    # simple if-rules over several results
    status = {r["key"]: r["status"] for r in rows if r.get("key")}
    names = {r["key"]: r["name"] for r in rows if r.get("key")}

    def is_(key, value):
        return status.get(key) == value

    found = []

    def add(title_en, title_ar, text_en, text_ar, keys):
        tests = [names[k] for k in keys if k in names]
        joined = ", ".join(tests)
        found.append({
            "title": {"en": title_en, "ar": title_ar},
            "text": {"en": text_en.format(tests=joined), "ar": text_ar.format(tests=joined)},
            "tests": tests,
        })

    iron_signs = [k for k in ("mcv", "mch", "ferritin", "iron") if is_(k, "Low")]
    if is_("hemoglobin", "Low") and iron_signs:
        add("Low hemoglobin with signs of low iron",
            "هيموجلوبين منخفض مع علامات على نقص الحديد",
            "These results are low together: {tests}. This combination is commonly seen when the body's iron stores are low, but there are other possible causes. Your doctor can work out the cause using your history and, if needed, more tests.",
            "هذه النتائج منخفضة معاً: {tests}. هذا التجمع يُرى كثيراً عندما يكون مخزون الحديد في الجسم منخفضاً، لكن توجد أسباب أخرى ممكنة. يستطيع طبيبك تحديد السبب بالاعتماد على تاريخك الصحي وبعض الفحوصات إن لزم.",
            ["hemoglobin"] + iron_signs)
    elif is_("ferritin", "Low"):
        add("Low ferritin with normal hemoglobin",
            "فيريتين منخفض مع هيموجلوبين غير منخفض",
            "{tests} is low while hemoglobin is not low. Ferritin shows iron stores and can fall before anemia appears, so it is worth discussing with your doctor.",
            "{tests} منخفض بينما الهيموجلوبين غير منخفض. الفيريتين يعكس مخزون الحديد وقد ينخفض قبل ظهور الأنيميا، لذلك يستحق النقاش مع طبيبك.",
            ["ferritin"])

    b_low = [k for k in ("vitamin_b12", "folate") if is_(k, "Low")]
    if b_low and (is_("mcv", "High") or is_("hemoglobin", "Low")):
        extra = [k for k in ("mcv", "hemoglobin") if is_(k, "High") or is_(k, "Low")]
        add("Low B12 or folate with blood count changes",
            "نقص B12 أو الفوليك مع تغيّر في صورة الدم",
            "These results appear together: {tests}. Low B12 or folate can be linked to changes in red blood cells. Your doctor can tell you whether this explains the blood count.",
            "ظهرت هذه النتائج معاً: {tests}. نقص B12 أو الفوليك قد يرتبط بتغيّر في كرات الدم الحمراء. يستطيع طبيبك أن يوضح هل يفسّر ذلك صورة الدم.",
            b_low + extra)

    sugar_high = [k for k in ("glucose_fasting", "glucose_random") if is_(k, "High")]
    if sugar_high and is_("hba1c", "High"):
        add("Both blood sugar and HbA1c are above range",
            "السكر والسكر التراكمي مرتفعان معاً",
            "{tests} are both above range. Together they suggest sugar has been higher than usual over recent weeks, but only your doctor can say what this means.",
            "{tests} مرتفعان معاً. هذا يشير إلى أن السكر كان أعلى من المعتاد خلال الأسابيع الأخيرة، لكن الطبيب وحده يستطيع تحديد المعنى.",
            sugar_high + ["hba1c"])
    elif is_("glucose_fasting", "High") and is_("hba1c", "Normal"):
        add("Raised sugar with a normal HbA1c",
            "سكر مرتفع مع سكر تراكمي طبيعي",
            "{tests} is above range but HbA1c is normal. A single raised reading can come from a recent meal, stress or illness, which is why doctors often repeat the test.",
            "{tests} مرتفع بينما السكر التراكمي طبيعي. قد تنتج قراءة واحدة مرتفعة عن وجبة حديثة أو توتر أو مرض، ولذلك يعيد الأطباء التحليل غالباً.",
            ["glucose_fasting", "hba1c"])
    elif is_("hba1c", "High") and not sugar_high:
        add("HbA1c above range",
            "السكر التراكمي مرتفع",
            "{tests} is above range. It reflects your average sugar over about 3 months, so your doctor may want to look at it together with a fasting sugar test.",
            "{tests} مرتفع. هو يعكس متوسط السكر خلال نحو 3 شهور، لذلك قد يرغب طبيبك في قراءته مع تحليل سكر صائم.",
            ["hba1c"])

    fat_high = [k for k in ("total_cholesterol", "triglycerides") if is_(k, "High")]
    if is_("ldl", "High") and fat_high:
        add("Several blood fats are above range",
            "عدة دهون في الدم مرتفعة",
            "{tests} are above range together. Doctors look at the whole lipid profile, together with age, blood pressure, smoking and family history, to judge heart risk.",
            "{tests} مرتفعة معاً. يقرأ الأطباء صورة الدهون كاملة مع العمر وضغط الدم والتدخين والتاريخ العائلي لتقدير مخاطر القلب.",
            ["ldl"] + fat_high)
    if is_("triglycerides", "High") and is_("hdl", "Low"):
        add("High triglycerides with low HDL",
            "دهون ثلاثية مرتفعة مع HDL منخفض",
            "{tests} move in this direction together. This pairing is often linked with sugar handling, weight and activity level, so it is a good point to discuss with your doctor.",
            "{tests} تتحرك في هذا الاتجاه معاً. هذا التزامن يرتبط كثيراً بتعامل الجسم مع السكر والوزن ومستوى النشاط، لذلك يستحق النقاش مع طبيبك.",
            ["triglycerides", "hdl"])

    if is_("alt", "High") and is_("ast", "High"):
        add("Both liver enzymes are above range",
            "إنزيما الكبد مرتفعان معاً",
            "{tests} are both above range. Common reasons include recent hard exercise, alcohol and some medicines, and your doctor may also want to check for liver conditions.",
            "{tests} مرتفعان معاً. من الأسباب الشائعة المجهود العنيف الأخير والكحول وبعض الأدوية، وقد يرغب طبيبك أيضاً في التحقق من أمراض الكبد.",
            ["alt", "ast"])
    if is_("alp", "High") and is_("ggt", "High"):
        add("ALP and GGT are both above range",
            "ALP وGGT مرتفعان معاً",
            "{tests} are both above range. When they rise together, the bile ducts or liver are more often involved than bone, but your doctor will decide what to check next.",
            "{tests} مرتفعان معاً. عندما يرتفعان معاً يكون الاحتمال الأكبر تأثر القنوات الصفراوية أو الكبد أكثر من العظام، لكن طبيبك هو من يحدد الخطوة التالية.",
            ["alp", "ggt"])

    if is_("creatinine", "High") and (is_("urea", "High") or is_("bun", "High") or is_("egfr", "Low")):
        keys = ["creatinine"] + [k for k in ("urea", "bun", "egfr") if is_(k, "High") or is_(k, "Low")]
        add("Kidney filtering markers are off together",
            "مؤشرات ترشيح الكلى متغيّرة معاً",
            "{tests} point in the same direction. Dehydration is a common temporary reason, but because these tests relate to the kidneys, it is worth discussing soon with your doctor.",
            "{tests} تتجه في الاتجاه نفسه. الجفاف سبب مؤقت شائع، لكن لأن هذه التحاليل مرتبطة بالكلى فمن المفيد مناقشتها مع طبيبك قريباً.",
            keys)

    if is_("tsh", "High") and is_("free_t4", "Low"):
        add("TSH high with low free T4",
            "TSH مرتفع مع T4 حر منخفض",
            "{tests} move in opposite directions. Doctors look for this pairing when checking for an underactive thyroid, and only they can confirm it.",
            "{tests} تتحرك في اتجاهين متعاكسين. يبحث الأطباء عن هذا التزامن عند تقييم كسل الغدة الدرقية، والطبيب وحده يستطيع تأكيده.",
            ["tsh", "free_t4"])
    elif is_("tsh", "Low") and (is_("free_t4", "High") or is_("free_t3", "High")):
        keys = ["tsh"] + [k for k in ("free_t4", "free_t3") if is_(k, "High")]
        add("TSH low with high thyroid hormones",
            "TSH منخفض مع ارتفاع هرمونات الغدة",
            "{tests} move in opposite directions. Doctors look for this pairing when checking for an overactive thyroid, and only they can confirm it.",
            "{tests} تتحرك في اتجاهين متعاكسين. يبحث الأطباء عن هذا التزامن عند تقييم النشاط الزائد للغدة الدرقية، والطبيب وحده يستطيع تأكيده.",
            keys)

    inflamed = [k for k in ("crp", "hs_crp", "esr", "wbc") if is_(k, "High")]
    if len(inflamed) >= 2:
        add("Several inflammation markers are raised",
            "عدة مؤشرات التهاب مرتفعة",
            "{tests} are all above range. Infection and inflammation are common reasons, and these markers often settle once the cause passes.",
            "{tests} مرتفعة معاً. العدوى والالتهاب من الأسباب الشائعة، وغالباً تعود هذه المؤشرات للانخفاض عند زوال السبب.",
            inflamed)

    if is_("vitamin_d", "Low") and (is_("calcium", "Low") or is_("phosphorus", "Low") or is_("alp", "High")):
        keys = ["vitamin_d"] + [k for k in ("calcium", "phosphorus") if is_(k, "Low")] + (["alp"] if is_("alp", "High") else [])
        add("Low vitamin D with bone-related results",
            "فيتامين د منخفض مع نتائج مرتبطة بالعظام",
            "{tests} are outside range together. Vitamin D helps the body absorb calcium for bones, so your doctor may want to look at them as a group.",
            "{tests} خارج المدى معاً. فيتامين د يساعد الجسم على امتصاص الكالسيوم للعظام، لذلك قد يرغب طبيبك في قراءتها كمجموعة.",
            keys)

    return found


Writing medical_explainer/medical_core.py


## 5) Engine
Loads the quantized model and the FAISS index, and builds the chains (extraction, explanation, summary) with their output parsers.


In [5]:
%%writefile medical_explainer/engine.py
import os
import threading
from typing import Any, List, Optional

import torch
from langchain_classic.chains import LLMChain
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import medical_core as core

MODEL_ID = os.environ.get("MODEL_ID", "mistralai/Mistral-Nemo-Instruct-2407")
EMBEDDING_ID = os.environ.get("EMBEDDING_ID", "sentence-transformers/all-MiniLM-L6-v2")
MAX_REPORT_CHARS = 6000
MAX_EXPLAINED = 10
TEST_TAGS = core.SECTION_TAGS

EXTRACT_TEMPLATE = """You are a careful reader of medical laboratory reports.
Extract every laboratory test result from the report below.

Rules:
- Copy the numbers exactly as written in the report.
- Do not invent tests and do not guess missing values.
- Ignore patient details, dates, doctor names and page headers.
- Use the standard English name of each test, even if the report is written in another language.
- If the report prints a reference range for a test, copy it; otherwise leave it empty.

{format_instructions}

Report:
{report}
"""

EXPLAIN_TEMPLATE = """You are a friendly assistant that explains one laboratory result to a patient in simple, calm language.

Rules:
- Use only the facts and the reference context below.
- Do not diagnose, do not say the patient has or does not have a disease, and do not suggest medicines, supplements or doses.
- Mention common and non-serious reasons first when the context lists them.
- Write the content of every section in {language}, but keep the section labels exactly as written in English capital letters.
- Use exactly the sections below, each starting on a new line with its label followed by a colon.

{sections_spec}

Patient: {patient}
Test: {test}
Result: {value} {unit}
Status: {status} (reference range {range})
Distance from the range: {distance}
Related results in the same report: {related}

Reference context:
{context}
"""

SPEC_BASE = (
    "MEANING: 1 to 2 sentences on what this test measures and why doctors order it.\n"
    "YOUR RESULT: 2 sentences saying whether the result is above or below the range, by how much, and what that usually means in plain words.\n"
    "COMMON REASONS: 2 to 3 sentences with common reasons for this kind of result, non-serious ones first, and a note that results can also be temporary.\n"
)
SPEC_RELATED = "RELATED RESULTS: 1 sentence on what the related results from this report suggest when read together with this one.\n"
SPEC_ASK = "ASK YOUR DOCTOR: 1 specific question the patient can ask about this result.\n"

SUMMARY_TEMPLATE = """You are a careful assistant that summarizes laboratory results for a patient.

Rules:
- Use only the results listed below.
- Do not diagnose and do not suggest treatment or medicines.
- Write every text value in {language}.
- The summary should be 4 to 5 short sentences and should say which results matter most to discuss first.

Patient: {patient}
Number of tests reviewed: {total}
Results outside the reference range:
{abnormal}
Patterns already found by rules (mention them only if they are listed):
{patterns}

{format_instructions}
"""


class LocalLLM(LLM):
    tokenizer: Any = None
    model: Any = None
    lock: Any = None
    max_new_tokens: int = 256

    @property
    def _llm_type(self) -> str:
        return "local_hf_4bit"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Any = None, **kwargs: Any) -> str:
        messages = [{"role": "user", "content": prompt}]
        encoded = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
        input_ids = encoded.input_ids if hasattr(encoded, "input_ids") else encoded
        input_ids = input_ids.to(self.model.device)
        with self.lock:
            with torch.no_grad():
                output = self.model.generate(
                    input_ids=input_ids,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
        return self.tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()


def entry_to_context(entry):
    low, high = core.kb_range(entry, None)
    return (
        f"{entry['name']} (also called: {', '.join(entry['aliases'][:4])}). "
        f"Category: {entry['category']}. Typical adult range: {core.format_range(low, high)} {entry['unit']}. "
        f"What it measures: {entry['measures']} "
        f"Why doctors order it: {entry.get('why', '')} "
        f"If low: {entry['low']} "
        f"If high: {entry['high']} "
        f"What can affect the result: {entry.get('prep', '')}"
    )


def entry_to_text(entry):
    low, high = core.kb_range(entry, None)
    return (
        f"{entry['name']} (also called: {', '.join(entry['aliases'][:4])}). "
        f"Category: {entry['category']}. Typical adult range: {core.format_range(low, high)} {entry['unit']}. "
        f"What it measures: {entry['measures']} "
        f"If low: {entry['low']} "
        f"If high: {entry['high']}"
    )


def build_documents(kb):
    return [
        Document(page_content=entry_to_text(entry), metadata={"key": entry["key"]})
        for entry in kb["entries"]
    ]


def load_resources(kb_path):
    kb = core.load_knowledge_base(kb_path)

    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_ID,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vectordb = FAISS.from_documents(build_documents(kb), embeddings)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
    model.eval()
    lock = threading.Lock()

    def make_llm(max_new_tokens):
        return LocalLLM(tokenizer=tokenizer, model=model, lock=lock, max_new_tokens=max_new_tokens)

    extract_parser = StructuredOutputParser.from_response_schemas([
        ResponseSchema(
            name="tests",
            description=(
                "A list of all lab test results found in the report. Each item is an object with the keys "
                "name (standard English test name), value (number only), unit, and reference_range "
                "(the range printed in the report, or an empty string)."
            ),
        )
    ])
    summary_parser = StructuredOutputParser.from_response_schemas([
        ResponseSchema(name="summary", description="A short overall summary of the results in 3 to 4 sentences, written for the patient."),
        ResponseSchema(name="questions", description="A list of 3 to 5 short questions the patient can ask their doctor."),
    ])

    extract_chain = LLMChain(
        llm=make_llm(1500),
        prompt=PromptTemplate(template=EXTRACT_TEMPLATE, input_variables=["report", "format_instructions"]),
    )
    explain_chain = LLMChain(
        llm=make_llm(560),
        prompt=PromptTemplate(
            template=EXPLAIN_TEMPLATE,
            input_variables=[
                "language", "patient", "test", "value", "unit", "status", "range",
                "distance", "related", "context", "sections_spec",
            ],
        ),
    )
    summary_chain = LLMChain(
        llm=make_llm(700),
        prompt=PromptTemplate(
            template=SUMMARY_TEMPLATE,
            input_variables=["language", "patient", "total", "abnormal", "patterns", "format_instructions"],
        ),
    )

    return {
        "kb": kb,
        "vectordb": vectordb,
        "extract_chain": extract_chain,
        "extract_parser": extract_parser,
        "explain_chain": explain_chain,
        "summary_chain": summary_chain,
        "summary_parser": summary_parser,
    }


def extract_with_llm(resources, report_text):
    parser = resources["extract_parser"]
    raw = resources["extract_chain"].invoke({
        "report": report_text[:MAX_REPORT_CHARS],
        "format_instructions": parser.get_format_instructions(),
    })["text"]
    block = core.extract_json_block(raw)
    try:
        parsed = parser.parse("```json\n" + block + "\n```")
    except Exception:
        return []
    items = parsed.get("tests", [])
    if isinstance(items, dict):
        items = [items]
    if not isinstance(items, list):
        return []
    return [item for item in items if isinstance(item, dict)]


def retrieve_context(resources, row):
    if not row["key"]:
        return "No reference entry is available for this test. Give only a short, general explanation and say that it is general information."
    kb = resources["kb"]
    texts = [entry_to_context(kb["by_key"][row["key"]])]
    seen = {row["key"]}
    docs = resources["vectordb"].similarity_search(f"{row['name']} {row['status']}", k=3)
    for doc in docs:
        key = doc.metadata.get("key")
        if key not in seen:
            texts.append(doc.page_content)
            seen.add(key)
        if len(texts) >= 3:
            break
    return "\n\n".join(texts)


def explain_row(resources, row, patient, language, related):
    related_line = core.format_related(related)
    spec = SPEC_BASE + (SPEC_RELATED if related_line else "") + SPEC_ASK
    text = resources["explain_chain"].invoke({
        "language": language,
        "patient": patient,
        "test": row["name"],
        "value": f"{row['value']:g}",
        "unit": row["unit"],
        "status": row["status"],
        "range": row["range"] or "not available",
        "distance": core.describe_distance(row, "English") or "not available",
        "related": related_line or "none",
        "context": retrieve_context(resources, row),
        "sections_spec": spec,
    })["text"].strip()
    return text, core.parse_sections(text)


def summarize_rows(resources, rows, patient, language, patterns=None):
    abnormal = [row for row in rows if row["status"] in ("Low", "High")]
    if not abnormal:
        return core.fallback_summary(rows), core.default_questions()
    lines = "\n".join(
        f"- {row['name']}: {row['value']:g} {row['unit']} ({row['status']}, reference {row['range']})"
        for row in abnormal
    )
    parser = resources["summary_parser"]
    raw = resources["summary_chain"].invoke({
        "language": language,
        "patient": patient,
        "total": str(len(rows)),
        "abnormal": lines,
        "patterns": "; ".join(p["title"]["en"] for p in (patterns or [])) or "none",
        "format_instructions": parser.get_format_instructions(),
    })["text"]
    try:
        parsed = parser.parse("```json\n" + core.extract_json_block(raw) + "\n```")
        summary = str(parsed.get("summary", "")).strip()
        questions = parsed.get("questions", [])
        if isinstance(questions, str):
            questions = [line.strip("-* \t") for line in questions.splitlines() if line.strip()]
        questions = [str(question).strip() for question in questions if str(question).strip()]
        if summary and questions:
            return summary, questions
    except Exception:
        pass
    return core.fallback_summary(rows), core.default_questions()


def analyze_report(resources, report_text, sex, age, language, mode, progress=None, max_explained=MAX_EXPLAINED):
    def step(message, fraction):
        if progress:
            progress(message, fraction)

    kb = resources["kb"]
    patient = f"{sex or 'sex not specified'}, {age} years old"

    step("Reading the report...", 0.05)
    items = []
    if mode in ("auto", "rules"):
        items = core.regex_extract(report_text, kb)
    if mode == "ai" or (mode == "auto" and len(items) < 3):
        step("Asking the AI model to extract the test results...", 0.15)
        ai_items = core.filter_by_text(extract_with_llm(resources, report_text), report_text)
        items = ai_items if mode == "ai" else items + ai_items

    rows = core.evaluate_tests(items, kb, sex)
    if not rows:
        return {"error": "No laboratory test results could be found in this report. Try pasting the text of the results table, or change the extraction mode."}

    abnormal_indexes = [index for index, row in enumerate(rows) if row["status"] in ("Low", "High")]
    abnormal_indexes.sort(key=lambda index: (not rows[index]["urgent"], -(rows[index]["deviation_pct"] or 0), index))
    selected = abnormal_indexes[:max_explained]
    patterns = core.detect_patterns(rows)
    for position, index in enumerate(selected, start=1):
        row = rows[index]
        step(f"Explaining {row['name']} ({position}/{len(selected)})...", 0.25 + 0.6 * position / max(len(selected), 1))
        related = core.related_results(row, rows, kb)
        text, sections = explain_row(resources, row, patient, language, related)
        rows[index]["explanation"] = text
        rows[index]["sections"] = sections

    step("Writing the overall summary...", 0.92)
    summary, questions = summarize_rows(resources, rows, patient, language, patterns)
    step("Done", 1.0)

    return {
        "rows": rows,
        "counts": core.summarize_counts(rows),
        "summary": summary,
        "questions": questions,
        "patterns": patterns,
        "language": language,
        "patient": patient,
        "skipped_explanations": max(len(abnormal_indexes) - len(selected), 0),
    }

Writing medical_explainer/engine.py


## 5b) Nearby doctors and hospitals


In [6]:
%%writefile medical_explainer/locator.py
import math
import re
import time
from urllib.parse import quote

import requests

USER_AGENT = "MedicalReportExplainer/1.0 (educational project)"
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
OVERPASS_URLS = (
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://overpass.openstreetmap.fr/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
)
TIMEOUT = 30
OVERPASS_TIMEOUT = 25
OVERPASS_ROUNDS = 2
MAX_ELEMENTS = 400
MIN_RESULTS = 5
MAX_RADIUS_KM = 30

KIND_LABELS = {
    "hospital": {"en": "Hospital", "ar": "مستشفى"},
    "clinic": {"en": "Clinic / medical center", "ar": "عيادة / مركز طبي"},
    "doctor": {"en": "Doctor", "ar": "طبيب"},
}


SPECIALTIES = {
    "internal": {
        "en": "Internal medicine (general physician)",
        "ar": "الباطنة (طبيب عام)",
        "search_en": "internal medicine doctor",
        "search_ar": "دكتور باطنة",
        "osm": {"internal", "internal_medicine", "general", "general_practice", "general_medicine", "family_medicine"},
    },
    "hematology": {
        "en": "Hematology (blood disorders)",
        "ar": "أمراض الدم",
        "search_en": "hematologist",
        "search_ar": "دكتور أمراض دم",
        "osm": {"haematology", "hematology"},
    },
    "endocrinology": {
        "en": "Endocrinology (diabetes and thyroid)",
        "ar": "الغدد الصماء والسكر",
        "search_en": "endocrinologist",
        "search_ar": "دكتور غدد صماء وسكر",
        "osm": {"endocrinology", "diabetology", "diabetes"},
    },
    "cardiology": {
        "en": "Cardiology (heart and blood fats)",
        "ar": "القلب",
        "search_en": "cardiologist",
        "search_ar": "دكتور قلب",
        "osm": {"cardiology"},
    },
    "hepatology": {
        "en": "Liver and digestive system",
        "ar": "الكبد والجهاز الهضمي",
        "search_en": "gastroenterologist hepatologist",
        "search_ar": "دكتور كبد وجهاز هضمي",
        "osm": {"hepatology", "gastroenterology"},
    },
    "nephrology": {
        "en": "Nephrology (kidneys)",
        "ar": "الكلى",
        "search_en": "nephrologist",
        "search_ar": "دكتور كلى",
        "osm": {"nephrology"},
    },
}


CATEGORY_SPECIALTIES = {
    "Complete Blood Count": ["hematology"],
    "Blood Sugar": ["endocrinology"],
    "Thyroid": ["endocrinology"],
    "Lipid Profile": ["cardiology"],
    "Liver Function": ["hepatology"],
    "Kidney Function": ["nephrology"],
}

COORD_RE = re.compile(r"^\s*(-?\d{1,2}(?:\.\d+)?)\s*[,;\s]\s*(-?\d{1,3}(?:\.\d+)?)\s*$")


def parse_coordinates(text):
    # lets the user type coordinates directly, like 30.0131, 31.2089
    match = COORD_RE.match(str(text or ""))
    if not match:
        return None
    lat, lon = float(match.group(1)), float(match.group(2))
    if -90 <= lat <= 90 and -180 <= lon <= 180:
        return {"lat": lat, "lon": lon, "name": f"{lat:.4f}, {lon:.4f}"}
    return None


def geocode(query, language="English"):
    # city / area / address -> coordinates, None if nothing is found
    query = str(query or "").strip()
    if not query:
        return None
    direct = parse_coordinates(query)
    if direct:
        return direct
    response = requests.get(
        NOMINATIM_URL,
        params={
            "q": query,
            "format": "jsonv2",
            "limit": 1,
            "accept-language": "ar" if language == "Arabic" else "en",
        },
        headers={"User-Agent": USER_AGENT},
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    data = response.json()
    if not data:
        return None
    item = data[0]
    return {"lat": float(item["lat"]), "lon": float(item["lon"]), "name": item.get("display_name", query)}


def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    d_lat = p2 - p1
    d_lon = math.radians(lon2 - lon1)
    a = math.sin(d_lat / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(d_lon / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(a))


def build_query(lat, lon, radius_m):
    around = f"(around:{int(radius_m)},{lat:.5f},{lon:.5f})"
    selectors = (
        '["amenity"~"^(hospital|clinic|doctors)$"]',
        '["healthcare"~"^(hospital|clinic|doctor)$"]',
    )
    body = "".join(f"nwr{selector}{around};" for selector in selectors)
    return f"[out:json][timeout:20];({body});out center tags {MAX_ELEMENTS};"


def fetch_overpass(query):
   
    errors = []
    for attempt in range(OVERPASS_ROUNDS):
        for url in OVERPASS_URLS:
            host = url.split("/")[2]
            try:
                response = requests.post(url, data={"data": query}, headers={"User-Agent": USER_AGENT}, timeout=OVERPASS_TIMEOUT)
                response.raise_for_status()
                return response.json().get("elements", [])
            except Exception as error:
                errors.append(f"{host}: {type(error).__name__}")
        if attempt < OVERPASS_ROUNDS - 1:
            time.sleep(3)
    raise RuntimeError("Could not reach the map service (" + "; ".join(errors) + ")")


def kind_of(tags):
    values = {tags.get("amenity"), tags.get("healthcare")}
    if "hospital" in values:
        return "hospital"
    if "clinic" in values:
        return "clinic"
    if values & {"doctors", "doctor"}:
        return "doctor"
    return None


def safe_url(value):
    value = str(value or "").strip()
    if re.match(r"^https?://", value, flags=re.I):
        return value
    if value and " " not in value and "." in value and not value.lower().startswith(("javascript:", "data:")):
        return "https://" + value
    return ""


def pick_name(tags, language):
    keys = ("name:ar", "name", "name:en") if language == "Arabic" else ("name:en", "name", "name:ar")
    for key in keys:
        if tags.get(key):
            return tags[key].strip()
    return (tags.get("operator") or "").strip()


def build_address(tags):
    street = " ".join(part for part in (tags.get("addr:housenumber"), tags.get("addr:street")) if part)
    area = tags.get("addr:suburb") or tags.get("addr:neighbourhood") or tags.get("addr:district")
    parts = [street, area, tags.get("addr:city")]
    return ", ".join(part for part in parts if part)


def normalize_speciality(tags):
    raw = tags.get("healthcare:speciality") or tags.get("healthcare:specialty") or ""
    values = set()
    for item in str(raw).split(";"):
        item = re.sub(r"[\s\-]+", "_", item.strip().lower())
        if item:
            values.add(item)
    return values


def parse_elements(elements, lat, lon, language="English"):
    places = []
    seen = set()
    for element in elements:
        tags = element.get("tags") or {}
        kind = kind_of(tags)
        name = pick_name(tags, language)
        if not kind or not name:
            continue
        center = element.get("center") or {}
        p_lat = element.get("lat", center.get("lat"))
        p_lon = element.get("lon", center.get("lon"))
        if p_lat is None or p_lon is None:
            continue
        signature = (name.lower(), round(p_lat, 3), round(p_lon, 3))
        if signature in seen:
            continue
        seen.add(signature)
        places.append({
            "name": name,
            "kind": kind,
            "lat": float(p_lat),
            "lon": float(p_lon),
            "distance_km": haversine_km(lat, lon, p_lat, p_lon),
            "address": build_address(tags),
            "phone": tags.get("phone") or tags.get("contact:phone") or tags.get("contact:mobile") or "",
            "website": safe_url(tags.get("website") or tags.get("contact:website")),
            "hours": tags.get("opening_hours", ""),
            "emergency": tags.get("emergency") == "yes",
            "specialities": normalize_speciality(tags),
        })
    places.sort(key=lambda place: place["distance_km"])
    return places


def find_nearby(lat, lon, radius_km=5, language="English"):
    # search around the point, and double the radius if too few places are found
    # returns (places sorted by distance, radius used in km)
    radius = float(radius_km)
    places = []
    while True:
        elements = fetch_overpass(build_query(lat, lon, radius * 1000))
        places = parse_elements(elements, lat, lon, language)
        if len(places) >= MIN_RESULTS or radius >= MAX_RADIUS_KM:
            return places, radius
        radius = min(radius * 2, MAX_RADIUS_KM)


def suggest_specialties(rows):
    # which doctor types fit the abnormal groups of tests
    grouped = {}
    for row in rows:
        if row.get("status") not in ("Low", "High"):
            continue
        for key in CATEGORY_SPECIALTIES.get(row.get("category"), []):
            names = grouped.setdefault(key, [])
            if row["name"] not in names:
                names.append(row["name"])
    ordered = ["internal"] + [key for key, _ in sorted(grouped.items(), key=lambda item: -len(item[1]))]
    suggestions = []
    for key in ordered:
        info = SPECIALTIES[key]
        suggestions.append({
            "key": key,
            "en": info["en"],
            "ar": info["ar"],
            "search_en": info["search_en"],
            "search_ar": info["search_ar"],
            "osm": info["osm"],
            "tests": grouped.get(key, []),
        })
    return suggestions


def place_matches(place, suggestions):
    wanted = set()
    for suggestion in suggestions:
        wanted |= suggestion["osm"]
    return bool(place["specialities"] & wanted)


def maps_search_url(term, lat, lon):
    return f"https://www.google.com/maps/search/{quote(term)}/@{lat:.5f},{lon:.5f},14z"


def maps_directions_url(lat, lon):
    return f"https://www.google.com/maps/dir/?api=1&destination={lat:.6f},{lon:.6f}"


Writing medical_explainer/locator.py


## 6) Streamlit app

In [7]:
%%writefile medical_explainer/app.py
import html
import os

import pandas as pd
import streamlit as st
from pypdf import PdfReader

import engine
import locator
import medical_core as core

try:
    from streamlit_geolocation import streamlit_geolocation
except Exception:
    streamlit_geolocation = None

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
KB_PATH = os.path.join(BASE_DIR, "knowledge_base.json")

SAMPLE_REPORT = """Patient: Sample Patient        Age: 34        Sex: Female
Date: 2026-09-20

COMPLETE BLOOD COUNT
Test                    Result    Unit        Reference Range
Hemoglobin (Hb)         10.8      g/dL        12.0 - 15.5     L
Hematocrit (HCT)        34.1      %           36.0 - 46.0     L
RBC Count               4.3       x10^6/uL    4.1 - 5.1
MCV                     72.0      fL          80.0 - 100.0    L
MCH                     28.0      pg          27.0 - 33.0
WBC Count               7.1       x10^3/uL    4.5 - 11.0
Platelet Count          310       x10^3/uL    150 - 450

BLOOD SUGAR
Fasting Blood Sugar (FBS)   108   mg/dL   70 - 99    H
HbA1c                       5.9   %       < 5.7      H

LIPID PROFILE
Total Cholesterol       188       mg/dL       < 200
LDL Cholesterol         128       mg/dL       < 100      H
HDL Cholesterol         52        mg/dL       > 50
Triglycerides           120       mg/dL       < 150

LIVER AND KIDNEY
ALT (SGPT)              22        U/L         7 - 55
AST (SGOT)              25        U/L         8 - 43
Creatinine              0.8       mg/dL       0.59 - 1.04

THYROID, VITAMINS AND IRON
TSH                     2.3       mIU/L       0.4 - 4.0
Vitamin D, 25-Hydroxy   14.2      ng/mL       30 - 100        L
Vitamin B12             450       pg/mL       200 - 900
Ferritin                8         ng/mL       11 - 307        L
"""

MODE_LABELS = {
    "Auto (rules first, AI if needed)": "auto",
    "AI extraction (slower, handles messy text)": "ai",
    "Rules only (fastest)": "rules",
}

FLAGS = {"High": "\U0001F534", "Low": "\U0001F535", "Normal": "\U0001F7E2", "Unknown": "\u26AA"}

DISCLAIMERS = {
    "English": "For information only.",
    "Arabic": "للمعلومات فقط .",
}

LABELS = {
    "English": {
        "title": "Lab Report Explanation",
        "summary": "Summary",
        "patterns": "Patterns across your results",
        "patterns_hint": "Some results are easier to understand when read together. These notes are generated by fixed rules and describe common combinations only.",
        "results": "Your results",
        "distance_note": "The size of the distance from the range does not always show how important a result is. Some tests matter even when the difference is small.",
        "details": "What the abnormal results mean",
        "MEANING": "What this test is",
        "YOUR RESULT": "What your result means",
        "COMMON REASONS": "Common reasons",
        "RELATED RESULTS": "Related results in your report",
        "ASK YOUR DOCTOR": "Ask your doctor",
        "prep": "Before the test and what can affect it",
        "why": "Why doctors order it",
        "measures": "What it measures",
        "guide": "Guide to all your tests",
        "guide_hint": "Every test found in your report with what it measures, why it is ordered, and what can change it.",
        "unknown": "Results that could not be evaluated",
        "questions": "Questions to ask your doctor",
        "nearby": "Find a doctor or hospital near you",
        "nearby_hint": "Type your city or area, or share your location, to see nearby hospitals, clinics and doctors from OpenStreetMap.",
        "spec_title": "Doctors who usually review these results",
        "spec_first": "A general or internal medicine doctor is usually the best first stop and can refer you to a specialist if needed.",
        "spec_for": "for",
        "place_input": "Your city, area or address",
        "place_placeholder": "For example: Dokki, Giza, Egypt",
        "search": "Search",
        "gps_hint": "Or share your current location (your browser will ask for permission):",
        "gps_name": "Your device location",
        "radius": "Search distance (km)",
        "showing_near": "Showing places near",
        "kind_filter": "Show",
        "kind_all": "All",
        "kind_hospital": "Hospitals",
        "kind_clinic": "Clinics",
        "kind_doctor": "Doctors",
        "no_places": "No places of this type were found in OpenStreetMap near this location. Its coverage is incomplete in some areas, so please use the Google Maps links above.",
        "widened": "Few places were found nearby, so the search distance was widened to {km} km.",
        "maps_links": "Search on Google Maps",
        "directions": "Directions",
        "website": "Website",
        "phone": "Phone",
        "hours": "Hours",
        "matches": "Matches your results",
        "emergency_badge": "Emergency care",
        "away_fmt": "{d:.1f} km away",
        "nearby_error": "Could not reach the map service right now. Please try again in a moment, or use the Google Maps links.",
        "not_found": "That place was not found. Try adding the city and country, for example: Maadi, Cairo, Egypt.",
        "urgent_nearby": "Some of your results may need prompt attention, so hospitals are shown first. In an emergency, call your local emergency number now (for example 123 for an ambulance in Egypt).",
        "nearby_disclaimer": "The list comes from OpenStreetMap and may be incomplete. Call the place first to confirm the hours and the specialty.",
    },
    "Arabic": {
        "title": "شرح تقرير التحاليل",
        "summary": "الملخص",
        "patterns": "أنماط عبر نتائجك",
        "patterns_hint": "بعض النتائج يسهل فهمها عند قراءتها معاً. هذه الملاحظات تُنتج بقواعد ثابتة وتصف تجمعات شائعة فقط.",
        "results": "نتائجك",
        "distance_note": "حجم البعد عن المدى المرجعي لا يدل دائماً على أهمية النتيجة. بعض التحاليل تهم حتى لو كان الفرق صغيراً.",
        "details": "ماذا تعني النتائج غير الطبيعية",
        "MEANING": "ما هو هذا التحليل",
        "YOUR RESULT": "ماذا تعني نتيجتك",
        "COMMON REASONS": "أسباب شائعة",
        "RELATED RESULTS": "نتائج مرتبطة في تقريرك",
        "ASK YOUR DOCTOR": "اسأل طبيبك",
        "prep": "قبل التحليل وما قد يؤثر عليه",
        "why": "لماذا يطلبه الأطباء",
        "measures": "ماذا يقيس",
        "guide": "دليل لكل تحاليلك",
        "guide_hint": "كل تحليل وُجد في تقريرك مع ما يقيسه ولماذا يُطلب وما الذي قد يغيّر نتيجته.",
        "unknown": "نتائج تعذّر تقييمها",
        "questions": "أسئلة تسألها لطبيبك",
        "nearby": "ابحث عن دكتور أو مستشفى قريب منك",
        "nearby_hint": "اكتب مدينتك أو منطقتك، أو شارك موقعك، لتظهر لك مستشفيات وعيادات وأطباء قريبون من خرائط OpenStreetMap.",
        "spec_title": "الأطباء الذين يراجعون هذه النتائج عادةً",
        "spec_first": "طبيب الباطنة أو الطبيب العام هو غالباً أفضل بداية، وهو يحوّلك إلى أخصائي إذا لزم الأمر.",
        "spec_for": "بسبب",
        "place_input": "مدينتك أو منطقتك أو عنوانك",
        "place_placeholder": "مثال: الدقي، الجيزة، مصر",
        "search": "بحث",
        "gps_hint": "أو شارك موقعك الحالي (سيطلب المتصفح إذنك):",
        "gps_name": "موقع جهازك",
        "radius": "مسافة البحث (كم)",
        "showing_near": "الأماكن القريبة من",
        "kind_filter": "عرض",
        "kind_all": "الكل",
        "kind_hospital": "مستشفيات",
        "kind_clinic": "عيادات ومراكز",
        "kind_doctor": "أطباء",
        "no_places": "لم نجد أماكن من هذا النوع في OpenStreetMap قرب هذا الموقع. تغطية الخريطة ناقصة في بعض المناطق، لذلك استخدم روابط Google Maps بالأعلى.",
        "widened": "عدد الأماكن القريبة قليل، لذلك وسّعنا مسافة البحث إلى {km} كم.",
        "maps_links": "ابحث في Google Maps",
        "directions": "الاتجاهات",
        "website": "الموقع الإلكتروني",
        "phone": "التليفون",
        "hours": "المواعيد",
        "matches": "يتوافق مع نتائجك",
        "emergency_badge": "طوارئ",
        "away_fmt": "على بعد {d:.1f} كم",
        "nearby_error": "تعذّر الوصول إلى خدمة الخرائط الآن. حاول مرة أخرى بعد قليل أو استخدم روابط Google Maps.",
        "not_found": "لم نجد هذا المكان. جرّب إضافة المدينة والدولة، مثال: المعادي، القاهرة، مصر.",
        "urgent_nearby": "بعض نتائجك قد تحتاج اهتماماً سريعاً، لذلك تظهر المستشفيات أولاً. في حالات الطوارئ اتصل برقم الطوارئ فوراً (مثلاً 123 للإسعاف في مصر).",
        "nearby_disclaimer": "القائمة من OpenStreetMap وقد تكون ناقصة. اتصل بالمكان أولاً للتأكد من المواعيد والتخصص.",
    },
}

st.set_page_config(page_title="Medical Report Explainer", page_icon="\U0001FA7A", layout="wide")


@st.cache_resource(show_spinner="Loading the AI model and the knowledge base. The first run takes a few minutes...")
def get_resources():
    return engine.load_resources(KB_PATH)


def read_pdf(uploaded_file):
    reader = PdfReader(uploaded_file)
    return "\n".join((page.extract_text() or "") for page in reader.pages)


def load_sample():
    st.session_state["report_text"] = SAMPLE_REPORT


def show_text(text, rtl):
    if rtl:
        safe = html.escape(text).replace("\n", "<br>")
        st.markdown(f'<div dir="rtl" style="text-align:right">{safe}</div>', unsafe_allow_html=True)
    else:
        st.write(text)


def show_block(title, text, rtl):
    # bold title then a paragraph, right-to-left for Arabic
    if rtl:
        safe = html.escape(text).replace("\n", "<br>")
        st.markdown(
            f'<div dir="rtl" style="text-align:right"><b>{html.escape(title)}</b><br>{safe}</div>',
            unsafe_allow_html=True,
        )
    else:
        st.markdown(f"**{title}**")
        st.write(text)


def status_word(status, language):
    return core.STATUS_AR.get(status, status) if language == "Arabic" else status


def pick(row, field, language):
    # use the _ar field when the language is Arabic
    if language == "Arabic" and row.get(field + "_ar"):
        return row[field + "_ar"]
    return row.get(field, "")


def section_items(row, labels):
    return [(tag, row["sections"][tag]) for tag in core.SECTION_TAGS if tag in row.get("sections", {})]


def build_markdown(result):
    language = result["language"]
    labels = LABELS[language]
    lines = [f"# {labels['title']}", "", f"Patient: {result['patient']}", "", f"## {labels['summary']}", "", result["summary"], ""]
    if result.get("patterns"):
        key = "ar" if language == "Arabic" else "en"
        lines += [f"## {labels['patterns']}", ""]
        for pattern in result["patterns"]:
            lines += [f"### {pattern['title'][key]}", "", pattern["text"][key], ""]
    lines += [f"## {labels['results']}", "", "| Test | Result | Reference range | Status | Distance |", "|---|---|---|---|---|"]
    for row in result["rows"]:
        lines.append(
            f"| {row['name']} | {row['value']:g} {row['unit']} | {row['range']} | {status_word(row['status'], language)} | {core.short_distance(row)} |"
        )
    lines.append("")
    explained = [row for row in result["rows"] if row["explanation"]]
    if explained:
        lines += [f"## {labels['details']}", ""]
        for row in explained:
            lines += [f"### {row['name']} ({status_word(row['status'], language)})", ""]
            distance = core.describe_distance(row, language)
            if distance:
                lines += [distance, ""]
            items = section_items(row, labels)
            if items:
                for tag, text in items:
                    lines += [f"**{labels[tag]}**", "", text, ""]
            else:
                lines += [row["explanation"], ""]
            prep = pick(row, "prep", language)
            if prep:
                lines += [f"**{labels['prep']}**", "", prep, ""]
    guide_rows = [row for row in result["rows"] if row["in_knowledge_base"]]
    if guide_rows:
        lines += [f"## {labels['guide']}", ""]
        for row in guide_rows:
            lines += [f"### {row['name']} ({status_word(row['status'], language)})", ""]
            for field, label in (("measures", "measures"), ("why", "why"), ("prep", "prep")):
                text = pick(row, field, language)
                if text:
                    lines += [f"- **{labels[label]}:** {text}"]
            lines.append("")
    lines += [f"## {labels['questions']}", ""]
    lines += [f"- {question}" for question in result["questions"]]
    lines += ["", "---", DISCLAIMERS[language]]
    return "\n".join(lines)


def html_block(inner, rtl):
    direction = "rtl" if rtl else "ltr"
    align = "right" if rtl else "left"
    st.markdown(
        f'<div dir="{direction}" style="text-align:{align}; margin-bottom:0.8rem">{inner}</div>',
        unsafe_allow_html=True,
    )


def html_link(url, label):
    return f'<a href="{html.escape(url, quote=True)}" target="_blank" rel="noopener noreferrer">{html.escape(label)}</a>'


@st.cache_data(ttl=3600, show_spinner=False)
def cached_geocode(query, language):
    return locator.geocode(query, language)


@st.cache_data(ttl=1800, show_spinner=False)
def cached_nearby(lat, lon, radius_km, language):
    return locator.find_nearby(lat, lon, radius_km, language)


def place_html(place, labels, rtl, matched):
    esc = html.escape
    ui = "ar" if rtl else "en"
    badges = [locator.KIND_LABELS[place["kind"]][ui]]
    if place["emergency"]:
        badges.append(labels["emergency_badge"])
    if matched:
        badges.append(labels["matches"])
    badge_text = esc(" | ".join(badges))
    lines = [f"<b>{esc(place['name'])}</b> &nbsp; <small>{badge_text}</small>"]
    lines.append(esc(labels["away_fmt"].format(d=place["distance_km"])))
    if place["address"]:
        lines.append(esc(place["address"]))
    if place["phone"]:
        lines.append(f"{labels['phone']}: <bdi>{esc(place['phone'])}</bdi>")
    if place["hours"]:
        lines.append(f"{labels['hours']}: <bdi>{esc(place['hours'])}</bdi>")
    links = [html_link(locator.maps_directions_url(place["lat"], place["lon"]), labels["directions"])]
    if place["website"]:
        links.append(html_link(place["website"], labels["website"]))
    lines.append(" &nbsp;|&nbsp; ".join(links))
    return "<br>".join(lines)


def show_nearby(result):
    language = result["language"]
    rtl = language == "Arabic"
    labels = LABELS[language]
    ui = "ar" if rtl else "en"
    rows = result["rows"]
    urgent = any(row["urgent"] for row in rows)
    suggestions = locator.suggest_specialties(rows)

    st.divider()
    st.subheader(labels["nearby"])
    st.caption(labels["nearby_hint"])
    if urgent:
        st.error(labels["urgent_nearby"])

    spec_lines = []
    for item in suggestions:
        line = "&bull; <b>" + html.escape(item[ui]) + "</b>"
        if item["tests"]:
            line += " (" + labels["spec_for"] + ": " + html.escape(", ".join(item["tests"])) + ")"
        spec_lines.append(line)
    html_block(
        "<b>" + labels["spec_title"] + "</b><br>" + "<br>".join(spec_lines) + "<br><i>" + labels["spec_first"] + "</i>",
        rtl,
    )

    with st.form("place_form"):
        query = st.text_input(labels["place_input"], placeholder=labels["place_placeholder"])
        submitted = st.form_submit_button(labels["search"])
    if submitted and query.strip():
        try:
            found = cached_geocode(query.strip(), language)
        except Exception:
            found = None
            st.warning(labels["nearby_error"])
        else:
            if found:
                st.session_state["location"] = found
            else:
                st.warning(labels["not_found"])

    if streamlit_geolocation is not None:
        st.caption(labels["gps_hint"])
        gps = streamlit_geolocation()
        if isinstance(gps, dict) and gps.get("latitude") is not None and gps.get("longitude") is not None:
            coords = (round(float(gps["latitude"]), 5), round(float(gps["longitude"]), 5))
            if st.session_state.get("gps_applied") != coords:
                st.session_state["gps_applied"] = coords
                st.session_state["location"] = {"lat": coords[0], "lon": coords[1], "name": labels["gps_name"]}

    location = st.session_state.get("location")
    if not location:
        st.caption(labels["nearby_disclaimer"])
        return
    lat, lon = location["lat"], location["lon"]

    links = []
    for item in suggestions:
        term = item["search_ar"] if rtl else item["search_en"]
        links.append(html_link(locator.maps_search_url(term, lat, lon), item[ui]))
    html_block(
        labels["showing_near"] + ": <b>" + html.escape(location["name"]) + "</b><br><br><b>"
        + labels["maps_links"] + ":</b> " + " &nbsp;|&nbsp; ".join(links),
        rtl,
    )

    radius = st.slider(labels["radius"], min_value=1, max_value=30, value=5, key="nearby_radius")
    kinds = ["all", "hospital", "clinic", "doctor"]
    kind = st.radio(
        labels["kind_filter"], kinds, index=1 if urgent else 0,
        format_func=lambda option: labels["kind_" + option], horizontal=True, key="nearby_kind",
    )

    failed = False
    try:
        places, used_radius = cached_nearby(round(lat, 4), round(lon, 4), radius, language)
    except Exception:
        places, used_radius, failed = [], radius, True
        st.warning(labels["nearby_error"])
    if used_radius > radius:
        st.info(labels["widened"].format(km=f"{used_radius:g}"))

    shown = [place for place in places if kind == "all" or place["kind"] == kind]
    if urgent:
        shown.sort(key=lambda place: (not place["emergency"], place["distance_km"]))
    shown = shown[:12]

    if shown:
        points = [{"lat": place["lat"], "lon": place["lon"]} for place in shown] + [{"lat": lat, "lon": lon}]
        st.map(pd.DataFrame(points))
        for place in shown:
            html_block(place_html(place, labels, rtl, locator.place_matches(place, suggestions)), rtl)
    elif not failed:
        st.info(labels["no_places"])
    st.caption(labels["nearby_disclaimer"])


def show_result(result):
    language = result["language"]
    rtl = language == "Arabic"
    labels = LABELS[language]
    key = "ar" if rtl else "en"
    rows = result["rows"]
    counts = result["counts"]

    urgent = [row for row in rows if row["urgent"]]
    if urgent:
        names = ", ".join(row["name"] for row in urgent)
        st.error(
            f"These values are far outside the usual range and may need prompt medical attention: {names}. "
            "Please contact your doctor or a medical service soon."
        )

    columns = st.columns(5)
    columns[0].metric("Tests found", counts["total"])
    columns[1].metric("Normal", counts["normal"])
    columns[2].metric("High", counts["high"])
    columns[3].metric("Low", counts["low"])
    columns[4].metric("Not evaluated", counts["unknown"])

    st.subheader(labels["summary"])
    show_text(result["summary"], rtl)

    patterns = result.get("patterns") or []
    if patterns:
        st.subheader(labels["patterns"])
        st.caption(labels["patterns_hint"])
        for pattern in patterns:
            show_block(pattern["title"][key], pattern["text"][key], rtl)
            st.divider()

    st.subheader(labels["results"])
    table = pd.DataFrame([
        {
            "": FLAGS.get(row["status"], ""),
            "Test": row["name"],
            "Result": f"{row['value']:g}",
            "Unit": row["unit"],
            "Reference range": row["range"],
            "Status": row["status"],
            "Distance": core.short_distance(row),
            "Range from": row["range_source"],
        }
        for row in rows
    ])
    st.dataframe(table, use_container_width=True, hide_index=True)
    st.caption(labels["distance_note"])

    explained = [row for row in rows if row["explanation"]]
    if explained:
        st.subheader(labels["details"])
        for row in explained:
            label = f"{FLAGS.get(row['status'], '')} {row['name']}: {row['value']:g} {row['unit']} ({status_word(row['status'], language)})"
            with st.expander(label, expanded=True):
                distance = core.describe_distance(row, language)
                if distance:
                    show_text(distance, rtl)
                items = section_items(row, labels)
                if items:
                    for tag, text in items:
                        show_block(labels[tag], text, rtl)
                else:
                    show_text(row["explanation"], rtl)
                prep = pick(row, "prep", language)
                if prep:
                    show_block(labels["prep"], prep, rtl)
                if row["note"]:
                    st.caption(row["note"])
                if not row["in_knowledge_base"]:
                    st.caption("This test is not in the knowledge base, so the explanation is general.")
    if result.get("skipped_explanations"):
        st.info(f"{result['skipped_explanations']} more abnormal results were listed in the table without a detailed explanation. You can raise the number of detailed explanations in the sidebar.")

    guide_rows = [row for row in rows if row["in_knowledge_base"]]
    if guide_rows:
        with st.expander(labels["guide"]):
            st.caption(labels["guide_hint"])
            current_category = None
            for row in guide_rows:
                if row["category"] != current_category:
                    current_category = row["category"]
                    st.markdown(f"#### {current_category}")
                title = f"{FLAGS.get(row['status'], '')} {row['name']}: {row['value']:g} {row['unit']} ({status_word(row['status'], language)})"
                body = []
                for field, label in (("measures", "measures"), ("why", "why"), ("prep", "prep")):
                    text = pick(row, field, language)
                    if text:
                        body.append(f"{labels[label]}: {text}")
                show_block(title, "\n".join(body), rtl)
                st.write("")

    unknown = [row for row in rows if row["status"] == "Unknown"]
    if unknown:
        with st.expander(labels["unknown"]):
            for row in unknown:
                reason = row["note"] or "No reference range is available for this test."
                st.markdown(f"**{row['name']}**: {row['value']:g} {row['unit']}. {reason}")

    st.subheader(labels["questions"])
    if rtl:
        show_text("\n".join(f"- {question}" for question in result["questions"]), True)
    else:
        for question in result["questions"]:
            st.markdown(f"- {question}")

    show_nearby(result)

    st.download_button(
        "Download this explanation",
        data=build_markdown(result),
        file_name="lab_report_explanation.md",
        mime="text/markdown",
    )


st.title("\U0001FA7A Medical Report Explainer")
st.caption("Paste your lab report or upload a PDF, and get a simple explanation of every result.")

st.sidebar.header("Patient details")
sex_label = st.sidebar.selectbox("Sex", ["Not specified", "Female", "Male"])
age = st.sidebar.number_input("Age (years)", min_value=0, max_value=120, value=30)
language = st.sidebar.selectbox("Explanation language", ["English", "Arabic"])
mode_label = st.sidebar.selectbox("Extraction mode", list(MODE_LABELS.keys()))
max_explained = st.sidebar.slider(
    "Detailed explanations (abnormal tests)", min_value=3, max_value=15, value=10,
    help="Each detailed explanation takes the model a few seconds. Lower it for faster results.",
)
if age < 18:
    st.sidebar.warning("The built-in reference ranges are for adults. Rely on the ranges printed in your report.")

sex = {"Female": "female", "Male": "male"}.get(sex_label)
mode = MODE_LABELS[mode_label]

resources = get_resources()

st.session_state.setdefault("report_text", "")
st.session_state.setdefault("result", None)

uploaded = st.file_uploader("Upload a PDF report (text-based PDFs only)", type=["pdf"])
if uploaded is not None:
    signature = (uploaded.name, uploaded.size)
    if st.session_state.get("loaded_file") != signature:
        st.session_state["loaded_file"] = signature
        extracted = read_pdf(uploaded)
        if extracted.strip():
            st.session_state["report_text"] = extracted
        else:
            st.warning("No readable text was found in this PDF. It may be a scanned image. Please paste the text instead.")

st.button("Load a sample report", on_click=load_sample)
report_text = st.text_area("Report text", key="report_text", height=320, placeholder="Paste the text of your lab results here...")

if st.button("Analyze report", type="primary", disabled=not report_text.strip()):
    progress_bar = st.progress(0.0, text="Starting...")

    def on_progress(message, fraction):
        progress_bar.progress(min(max(fraction, 0.0), 1.0), text=message)

    try:
        outcome = engine.analyze_report(
            resources, report_text, sex, int(age), language, mode, on_progress, max_explained=max_explained
        )
    except Exception as error:
        outcome = {"error": f"Something went wrong while analyzing the report: {error}"}
    progress_bar.empty()
    st.session_state["result"] = outcome

result = st.session_state.get("result")
if result:
    if "error" in result:
        st.error(result["error"])
    else:
        show_result(result)

st.divider()
st.caption(DISCLAIMERS[language])


Writing medical_explainer/app.py


## 7) Test


In [8]:
import sys
sys.path.insert(0, "medical_explainer")

import medical_core as core

kb = core.load_knowledge_base("medical_explainer/knowledge_base.json")

sample = """Hemoglobin (Hb)    10.8   g/dL   12.0 - 15.5   L
MCV   72.0   fL   80.0 - 100.0   L
Fasting Blood Sugar (FBS)   108   mg/dL   70 - 99   H
HbA1c   5.9   %   < 5.7   H
Vitamin D, 25-Hydroxy   14.2   ng/mL   30 - 100   L
Ferritin   8   ng/mL   11 - 307   L
TSH   2.3   mIU/L   0.4 - 4.0"""

rows = core.evaluate_tests(core.regex_extract(sample, kb), kb, "female")
for row in rows:
    print(f"{row['name']:25} {row['value']:>6g} {row['unit']:8} range={row['range']:10} status={row['status']:7} distance={core.short_distance(row)}")

print("\n--- Distance sentence (English and Arabic) ---")
print(core.describe_distance(rows[0], "English"))
print(core.describe_distance(rows[0], "Arabic"))

print("\n--- Related results for Hemoglobin ---")
print(core.format_related(core.related_results(rows[0], rows, kb)))

print("\n--- Patterns found ---")
for pattern in core.detect_patterns(rows):
    print("*", pattern["title"]["en"])
    print("  ", pattern["text"]["en"])

print("\n--- Section parser ---")
demo = "MEANING: Hemoglobin carries oxygen.\n**YOUR RESULT:** It is lower than the range.\nCOMMON REASONS: Low iron is common.\nASK YOUR DOCTOR: Should I repeat the test?"
print(core.parse_sections(demo))


Hemoglobin                  10.8 g/dL     range=12 - 15.5  status=Low     distance=10% below
MCV                           72 fL       range=80 - 100   status=Low     distance=10% below
Fasting Blood Glucose        108 mg/dL    range=70 - 99    status=High    distance=9% above
HbA1c                        5.9 %        range=<= 5.7     status=High    distance=4% above
Vitamin D (25-OH)           14.2 ng/mL    range=30 - 100   status=Low     distance=53% below
Ferritin                       8 ng/mL    range=11 - 307   status=Low     distance=27% below
TSH                          2.3 mIU/L    range=0.4 - 4    status=Normal  distance=

--- Distance sentence (English and Arabic) ---
The result is about 10% below the lower limit of the reference range (a small distance from the range).
النتيجة أقل من الحد الأدنى للمدى المرجعي بنحو 10% (مسافة صغيرة عن المدى).

--- Related results for Hemoglobin ---
MCV: 72 fL (Low); Ferritin: 8 ng/mL (Low)

--- Patterns found ---
* Low hemoglobin with signs 

## 7b) Test


In [9]:
import re
import sys
sys.path.insert(0, "medical_explainer")

import locator
import medical_core as core

kb = core.load_knowledge_base("medical_explainer/knowledge_base.json")
sample = """Hemoglobin (Hb)    10.8   g/dL   12.0 - 15.5   L
MCV   72.0   fL   80.0 - 100.0   L
Fasting Blood Sugar (FBS)   108   mg/dL   70 - 99   H
HbA1c   5.9   %   < 5.7   H
LDL Cholesterol   128   mg/dL   < 100   H
TSH   2.3   mIU/L   0.4 - 4.0"""
rows = core.evaluate_tests(core.regex_extract(sample, kb), kb, "female")

print("--- Suggested doctor types ---")
suggestions = locator.suggest_specialties(rows)
for item in suggestions:
    print("*", item["en"], "|", item["ar"], "|", ", ".join(item["tests"]) or "first stop")

# A fake Overpass answer, so no internet is needed for this check
lat, lon = 30.0131, 31.2089
elements = [
    {"type": "node", "lat": 30.0200, "lon": 31.2100, "tags": {"amenity": "hospital", "name": "Demo Hospital", "emergency": "yes", "website": "javascript:alert(1)"}},
    {"type": "way", "center": {"lat": 30.0140, "lon": 31.2090}, "tags": {"amenity": "clinic", "name": "Demo Blood Clinic", "healthcare:speciality": "haematology"}},
    {"type": "node", "lat": 30.0140, "lon": 31.2090, "tags": {"healthcare": "clinic", "name": "Demo Blood Clinic"}},
    {"type": "node", "lat": 30.0500, "lon": 31.2500, "tags": {"amenity": "doctors"}},
    {"type": "node", "lat": 30.0300, "lon": 31.2200, "tags": {"amenity": "pharmacy", "name": "Not a doctor"}},
]
places = locator.parse_elements(elements, lat, lon)
print("\n--- Parsed places (nearest first) ---")
for place in places:
    print(f"{place['distance_km']:.2f} km | {place['kind']:8} | {place['name']} | website={place['website']!r} | matches={locator.place_matches(place, suggestions)}")

assert [place["name"] for place in places] == ["Demo Blood Clinic", "Demo Hospital"]
assert places[1]["website"] == ""
print("\nGoogle Maps link:", locator.maps_search_url("endocrinologist", lat, lon))
print("OK")


--- Suggested doctor types ---
* Internal medicine (general physician) | الباطنة (طبيب عام) | first stop
* Hematology (blood disorders) | أمراض الدم | Hemoglobin, MCV
* Endocrinology (diabetes and thyroid) | الغدد الصماء والسكر | Fasting Blood Glucose, HbA1c
* Cardiology (heart and blood fats) | القلب | LDL Cholesterol

--- Parsed places (nearest first) ---
0.10 km | clinic   | Demo Blood Clinic | website='' | matches=True
0.77 km | hospital | Demo Hospital | website='' | matches=False

Google Maps link: https://www.google.com/maps/search/endocrinologist/@30.01310,31.20890,14z
OK


## 7c) Test the locator with real internet


In [10]:
import sys
sys.path.insert(0, "medical_explainer")
import locator

place = locator.geocode("Dokki, Giza, Egypt")
print("Found:", place)
try:
    places, radius = locator.find_nearby(place["lat"], place["lon"], 5)
    print(f"{len(places)} places within {radius:g} km")
    for item in places[:8]:
        print(f"{item['distance_km']:.1f} km | {item['kind']:8} | {item['name']}")
except RuntimeError as error:
    print("Map service is busy right now, run this cell again in a minute.")
    print(error)


Found: {'lat': 30.0388955, 'lon': 31.2125563, 'name': 'Dokki, Giza, Aj Jiza, 12611, Egypt'}
145 places within 5 km
0.1 km | doctor   | El Fouly ENT Clinic دكتور محمود الفولى أنف و أذن
0.2 km | hospital | 6 October Hospital
0.3 km | clinic   | Dokki scan
0.3 km | hospital | Finey Hospital
0.3 km | hospital | Finy hospital
0.4 km | clinic   | عيادة الدكتور "الأيمن فتحي حسين"
0.4 km | hospital | El Shabrawishy
0.6 km | clinic   | مستشفى ستار


## 8) Run the app
Starts Streamlit and opens a public ngrok link.


In [11]:
import os
import subprocess
import sys
import time

from pyngrok import ngrok, conf

MODEL_ID = "mistralai/Mistral-Nemo-Instruct-2407"
PORT = 8501

try:
    from kaggle_secrets import UserSecretsClient
    NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = "3KJd28wgW5xNiE6MfNYoZtHp5UQ_3atfTbgCNncnvnAUyDV3N"

if NGROK_TOKEN == "ADD_NGROK_TOKEN":
    raise ValueError("Add your ngrok token as a Kaggle secret named NGROK_TOKEN, or paste it into NGROK_TOKEN above.")

ngrok.kill()
conf.get_default().auth_token = NGROK_TOKEN

environment = {**os.environ, "MODEL_ID": MODEL_ID}
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "medical_explainer/app.py",
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    env=environment,
)
time.sleep(10)

public_url = ngrok.connect(PORT).public_url
print("Public URL:", public_url)
print("Open the link, then wait for the model to finish loading on the first visit.")

Public URL: https://immobile-shrine-varsity.ngrok-free.dev                                          
Open the link, then wait for the model to finish loading on the first visit.


## 9) Logs

In [12]:
!tail -n 40 streamlit.log

2026-10-08 10:42:55.020 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.42.228.222:8501

